# processforsampletext-microbetext

## pathseqtext

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm import tqdm # import tqdm

# textdirectorytextallcancer typetext
parent_dir = Path("/path/to/project/data_V3/cancers/").expanduser()

def get_matrix(cancer_type, Type):
    inbase_dir = parent_dir / cancer_type
    outbase_dir = Path(f"/path/to/project/results_V3/cancers_V3.1/{cancer_type}").expanduser()
    groups = ["Normal", "Tumor"] # this list can be adjusted as needed
    for group in groups:
        input_dir = inbase_dir / group / "abundance_pathseq_V3"
        output_dir = outbase_dir / '01.group_matrix'
        output_dir.mkdir(parents=True, exist_ok=True)

        old_eRPKM_df = pd.DataFrame()
        old_eRPKM_Normalized_df = pd.DataFrame()
        for file_path in input_dir.glob(f"*.txt"):
            sample = file_path.stem.replace(f"_rna_output.pathseq", "")
            abundance_file = input_dir / f"{sample}_rna_output.pathseq.txt"

            # readpathseqtxtfile
            abundance = pd.read_csv(abundance_file, sep='\t')

            # texttypecolumnfiltergenusorspeciesdata
            abundance_filtered = abundance[abundance['type'] == Type]
            eRPKM_df = abundance_filtered[["name", "score"]].copy()
            eRPKM_df.rename(columns={"score": sample}, inplace=True)
            eRPKM_Normalized_df = abundance_filtered[["name", "score_normalized"]].copy()
            eRPKM_Normalized_df.rename(columns={"score_normalized": sample}, inplace=True)

            if old_eRPKM_df.empty:
                old_eRPKM_df = eRPKM_df
            else:
                old_eRPKM_df = pd.merge(old_eRPKM_df, eRPKM_df, on="name", how="outer")
                if old_eRPKM_Normalized_df.empty:
                    old_eRPKM_Normalized_df = eRPKM_Normalized_df
                else:
                    old_eRPKM_Normalized_df = pd.merge(old_eRPKM_Normalized_df, eRPKM_Normalized_df, on="name", how="outer")
                    # text
                    old_eRPKM_df = old_eRPKM_df.fillna(0).infer_objects()
                    old_eRPKM_df.sort_values(by="name", inplace=True, ascending=True)

                    old_eRPKM_Normalized_df = old_eRPKM_Normalized_df.fillna(0).infer_objects()
                    old_eRPKM_Normalized_df.sort_values(by="name", inplace=True, ascending=True)

                    # textCSVfile
                    output_file = output_dir / f"{cancer_type}_{group}_{Type}_pathseq.csv"
                    old_eRPKM_df.to_csv(output_file, sep=',', index=False)

                    output_file = output_dir / f"{cancer_type}_{group}_{Type}_pathseq_Normalized.csv"
                    old_eRPKM_Normalized_df.to_csv(output_file, sep=',', index=False)

# textprocesscancer-type list
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# text tqdm text, textcancer type
for cancer_type_name in tqdm(cancer_types, desc="Processing Cancer Types"):
    cancer_type_path = parent_dir / cancer_type_name
    # checktextcancer typefiletextexists
    if cancer_type_path.exists() and cancer_type_path.is_dir():
        get_matrix(cancer_type_name, 'genus')
        get_matrix(cancer_type_name, 'species')
    else:
        print(f"Warning: {cancer_type_name} folder not found in {parent_dir}")

# calculatemicrobetext

## pan cancer

### species

In [ ]:
# species
import pandas as pd

# allcancer typetext
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# filepathtext
base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_species_pathseq.csv"

# readmicrobedatatext
# def get_nonzero_microbes(file_path):
# df = pd.read_csv(file_path)
# # removeallsampletextfor0rows, textwithtext0rows
# df_filtered = df[(df.iloc[:, 1:] != 0).any(axis=1)]
# microbes = df_filtered.iloc[:, 0]
# microbes = microbes[~microbes.str.lower().str.contains("virus")]
# return set(microbes)

def get_microbes_50pct_nonzero(file_path):
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = (sample_data.shape[1]) * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    # microbes = microbes[~microbes.str.lower().str.contains("virus")]
    return set(microbes)

# text normal and tumor microbetext
all_normal_microbes = set()
all_tumor_microbes = set()

for cancer in cancer_types:
    try:
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")

        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)

        all_normal_microbes.update(normal_microbes)
        all_tumor_microbes.update(tumor_microbes)
    except Exception as e:
        print(f" process {cancer} text: {e}")

# analyzetextaftertext
intersection = all_normal_microbes & all_tumor_microbes
union = all_normal_microbes | all_tumor_microbes
normal_unique = all_normal_microbes - all_tumor_microbes
tumor_unique = all_tumor_microbes - all_normal_microbes

# statistics
n_normal = len(all_normal_microbes)
n_tumor = len(all_tumor_microbes)
n_union = len(union)
n_intersection = len(intersection)

# output
print("===== allcancer typetextafterstatistics =====")
print(f"Total Normal microbetext: {n_normal}")
print(f"Total Tumor microbetext: {n_tumor}")
print(f"Union microbetext: {n_union}")
print(f"textcount: {n_intersection} ({n_intersection / n_union:.2%})")
print(f"Normal textwith: {len(normal_unique)} ({len(normal_unique) / n_union:.2%})")
print(f"Tumor textwith: {len(tumor_unique)} ({len(tumor_unique) / n_union:.2%})")



### genus

In [ ]:
# species
import pandas as pd

# allcancer typetext
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# filepathtext
base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_genus_pathseq.csv"

# readmicrobedatatext
# def get_nonzero_microbes(file_path):
# df = pd.read_csv(file_path)
# # removeallsampletextfor0rows, textwithtext0rows
# df_filtered = df[(df.iloc[:, 1:] != 0).any(axis=1)]
# microbes = df_filtered.iloc[:, 0]
# microbes = microbes[~microbes.str.lower().str.contains("virus")]
# return set(microbes)

def get_microbes_50pct_nonzero(file_path):
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = (sample_data.shape[1]) * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    # microbes = microbes[~microbes.str.lower().str.contains("virus")]
    return set(microbes)

# text normal and tumor microbetext
all_normal_microbes = set()
all_tumor_microbes = set()

for cancer in cancer_types:
    try:
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")

        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)

        all_normal_microbes.update(normal_microbes)
        all_tumor_microbes.update(tumor_microbes)
    except Exception as e:
        print(f" process {cancer} text: {e}")

# analyzetextaftertext
intersection = all_normal_microbes & all_tumor_microbes
union = all_normal_microbes | all_tumor_microbes
normal_unique = all_normal_microbes - all_tumor_microbes
tumor_unique = all_tumor_microbes - all_normal_microbes

# statistics
n_normal = len(all_normal_microbes)
n_tumor = len(all_tumor_microbes)
n_union = len(union)
n_intersection = len(intersection)

# output
print("===== allcancer typetextafterstatistics =====")
print(f"Total Normal microbetext: {n_normal}")
print(f"Total Tumor microbetext: {n_tumor}")
print(f"Union microbetext: {n_union}")
print(f"textcount: {n_intersection} ({n_intersection / n_union:.2%})")
print(f"Normal textwith: {len(normal_unique)} ({len(normal_unique) / n_union:.2%})")
print(f"Tumor textwith: {len(tumor_unique)} ({len(tumor_unique) / n_union:.2%})")



## per cancer

### species

In [ ]:
import pandas as pd
import os

# allcancer typetext
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# filepathtext
base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_species_pathseq.csv"

# readmicrobedatatext
# def get_nonzero_microbes(file_path):
# df = pd.read_csv(file_path)
# # removeallsampletextfor0rows, textwithtext0rows
# df_filtered = df[(df.iloc[:, 1:] != 0).any(axis=1)]
# return set(df_filtered.iloc[:, 0]) # returnmicrobetext

def get_microbes_50pct_nonzero(file_path):
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = (sample_data.shape[1]) * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    # microbes = microbes[~microbes.str.lower().str.contains("virus")]
    return set(microbes)
# textresults
results = []

for cancer in cancer_types:
    try:
        # buildpath
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")

        # readdata
        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)

        # text
        intersection = normal_microbes & tumor_microbes
        union = normal_microbes | tumor_microbes
        normal_unique = normal_microbes - tumor_microbes
        tumor_unique = tumor_microbes - normal_microbes

        # statisticsdata
        total_normal = len(normal_microbes)
        total_tumor = len(tumor_microbes)
        total_intersection = len(intersection)
        total_union = len(union)

        # resultstextlist
        results.append({
            "Cancer": cancer,
            "Normal_total": total_normal,
            "Tumor_total": total_tumor,
            "Union_total": total_union,
            "Intersection": total_intersection,
            "Normal_unique": len(normal_unique),
            "Tumor_unique": len(tumor_unique),
            "Normal_unique_pct": len(normal_unique) / total_union if total_union else 0,
            "Tumor_unique_pct": len(tumor_unique) / total_union if total_union else 0,
            "Intersection_pct": total_intersection / total_union if total_union else 0,
        })
    except Exception as e:
        print(f"process {cancer} text: {e}")

# outputresultstext
df_results = pd.DataFrame(results)
# print(df_results)

# optional: text CSV file
df_results.to_csv("/path/to/project/results_V3/summary/figure1AB/microbe_prevalence_pathseq_species_by_cancer.csv", index=False)

df_results


In [ ]:
import pandas as pd
import os

# allcancer typetext
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# filepathtext
base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_species_pathseq.csv"

# readallmicrobe(textfilter)
def get_all_microbes(file_path):
    df = pd.read_csv(file_path)
    return set(df.iloc[:, 0]) # return all microbe names from the first column

# textallcancer typemicrobe
all_species = set()

for cancer in cancer_types:
    try:
        # buildpath
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")
        # readdata
        normal_microbes = get_all_microbes(normal_file)
        tumor_microbes = get_all_microbes(tumor_file)
        # texttotext
        all_species.update(normal_microbes)
        all_species.update(tumor_microbes)
        print(f"OK {cancer}: Normal={len(normal_microbes)}, Tumor={len(tumor_microbes)}")
    except Exception as e:
        print(f"Failed process {cancer} text: {e}")

# outputtextspeciescount
total_species_count = len(all_species)
print(f"\n{'='*60}")
print(f"allcancer typeinunique speciestext: {total_species_count}")
print(f"{'='*60}")

# textsavespecieslist
# with open("/path/to/project/results_V3/summary/all_species_list.txt", "w") as f:
# for species in sorted(all_species):
# f.write(f"{species}\n")


### genus

In [ ]:
import pandas as pd
import os

# allcancer typetext
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# filepathtext
base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_genus_pathseq.csv"

# readmicrobedatatext
# def get_nonzero_microbes(file_path):
# df = pd.read_csv(file_path)
# # removeallsampletextfor0rows, textwithtext0rows
# df_filtered = df[(df.iloc[:, 1:] != 0).any(axis=1)]
# return set(df_filtered.iloc[:, 0]) # returnmicrobetext

def get_microbes_50pct_nonzero(file_path):
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = (sample_data.shape[1]) * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    # microbes = microbes[~microbes.str.lower().str.contains("virus")]
    return set(microbes)
# textresults
results = []

for cancer in cancer_types:
    try:
        # buildpath
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")

        # readdata
        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)

        # text
        intersection = normal_microbes & tumor_microbes
        union = normal_microbes | tumor_microbes
        normal_unique = normal_microbes - tumor_microbes
        tumor_unique = tumor_microbes - normal_microbes

        # statisticsdata
        total_normal = len(normal_microbes)
        total_tumor = len(tumor_microbes)
        total_intersection = len(intersection)
        total_union = len(union)

        # resultstextlist
        results.append({
            "Cancer": cancer,
            "Normal_total": total_normal,
            "Tumor_total": total_tumor,
            "Union_total": total_union,
            "Intersection": total_intersection,
            "Normal_unique": len(normal_unique),
            "Tumor_unique": len(tumor_unique),
            "Normal_unique_pct": len(normal_unique) / total_union if total_union else 0,
            "Tumor_unique_pct": len(tumor_unique) / total_union if total_union else 0,
            "Intersection_pct": total_intersection / total_union if total_union else 0,
        })
    except Exception as e:
        print(f"process {cancer} text: {e}")

# outputresultstext
df_results = pd.DataFrame(results)
# print(df_results)
df_results
# optional: text CSV file
df_results.to_csv("/path/to/project/results_V3/summary/figure1AB/microbe_prevalence_pathseq_genus_by_cancer.csv", index=False)
# print(df_results)
df_results

## R NR

In [ ]:
import pandas as pd
import gzip
from pathlib import Path
from tqdm import tqdm

# textsampletext
sample_groups = {
    'NR': ['B2', 'B3', 'B6', 'C6'],
    'R': ['B1', 'B4', 'B7', 'E5']
}

# textinputoutputpath - text
input_dirs = {
    'NR': Path("/path/to/project/data_V3/special/CRC/NR/abundance_pathseq_V3"),
    'R': Path("/path/to/project/data_V3/special/CRC/R/abundance_pathseq_V3")
}

output_dir = Path("/path/to/project/results_V3/special/CRC/01.group_matrix")
output_dir.mkdir(parents=True, exist_ok=True)

def get_matrix(Type_level, group_name, sample_list):
    """
    processtextandtextdata
    Type_level: 'genus' or 'species'
    group_name: 'R' or 'NR'
    sample_list: sample columntext
    """
    old_eRPKM_df = pd.DataFrame()
    old_eRPKM_Normalized_df = pd.DataFrame()
    # textinput directory
    input_dir = input_dirs[group_name]
    for sample in tqdm(sample_list, desc=f"Processing {group_name} {Type_level}"):
        # filenameformat: B1_rna_output.pathseq.txt.gz (text) or B1_rna_output.pathseq.txt (text)
        # text .txt.gz, textexiststext .txt
        file_path_gz = input_dir / f"{sample}_rna_output.pathseq.txt.gz"
        file_path_txt = input_dir / f"{sample}_rna_output.pathseq.txt"
        if file_path_gz.exists():
            file_path = file_path_gz
            open_func = lambda f: gzip.open(f, 'rt')
        elif file_path_txt.exists():
            file_path = file_path_txt
            open_func = lambda f: open(f, 'r')
        else:
            print(f"Warning: Neither {file_path_gz} nor {file_path_txt} found, skipping...")
            continue
        # readfile
        with open_func(file_path) as f:
            abundance = pd.read_csv(f, sep='\t')
            # texttypecolumnfiltergenusorspeciesdata
            abundance_filtered = abundance[abundance['type'] == Type_level]
            # extractscorecolumn
            eRPKM_df = abundance_filtered[["name", "score"]].copy()
            eRPKM_df.rename(columns={"score": sample}, inplace=True)
            # extractscore_normalizedcolumn
            eRPKM_Normalized_df = abundance_filtered[["name", "score_normalized"]].copy()
            eRPKM_Normalized_df.rename(columns={"score_normalized": sample}, inplace=True)
            # textdata
            if old_eRPKM_df.empty:
                old_eRPKM_df = eRPKM_df
            else:
                old_eRPKM_df = pd.merge(old_eRPKM_df, eRPKM_df, on="name", how="outer")
                if old_eRPKM_Normalized_df.empty:
                    old_eRPKM_Normalized_df = eRPKM_Normalized_df
                else:
                    old_eRPKM_Normalized_df = pd.merge(old_eRPKM_Normalized_df, eRPKM_Normalized_df, on="name", how="outer")
                    # text
                    old_eRPKM_df = old_eRPKM_df.fillna(0)
                    old_eRPKM_df.sort_values(by="name", inplace=True, ascending=True)
                    old_eRPKM_Normalized_df = old_eRPKM_Normalized_df.fillna(0)
                    old_eRPKM_Normalized_df.sort_values(by="name", inplace=True, ascending=True)
                    # savefile
                    output_file = output_dir / f"CRC_{group_name}_{Type_level}_pathseq.csv"
                    old_eRPKM_df.to_csv(output_file, sep=',', index=False)
                    print(f"Saved: {output_file}")
                    output_file_norm = output_dir / f"CRC_{group_name}_{Type_level}_pathseq_Normalized.csv"
                    old_eRPKM_Normalized_df.to_csv(output_file_norm, sep=',', index=False)
                    print(f"Saved: {output_file_norm}")
                    return old_eRPKM_df

def calculate_microbe_statistics(Type_level):
    """
    calculateRtextandNR-onlytextsharedmicrobecountandtext
    Type_level: 'genus' or 'species'
    """
    print(f"\n{'='*60}")
    print(f"Calculating statistics for {Type_level}")
    print(f"{'='*60}")
    # readRtextandNRtextdata
    r_file = output_dir / f"CRC_R_{Type_level}_pathseq.csv"
    nr_file = output_dir / f"CRC_NR_{Type_level}_pathseq.csv"
    r_df = pd.read_csv(r_file)
    nr_df = pd.read_csv(nr_file)
    # textwithtextmicrobe(textsampleinscore > 0)
    r_microbes = set(r_df[r_df.iloc[:, 1:].sum(axis=1) > 0]['name'])
    nr_microbes = set(nr_df[nr_df.iloc[:, 1:].sum(axis=1) > 0]['name'])
    # calculatetextwithandsharedmicrobe
    r_unique = r_microbes - nr_microbes # R-only
    nr_unique = nr_microbes - r_microbes # NR-only
    common = r_microbes & nr_microbes # shared
    # text
    total = len(r_unique) + len(nr_unique) + len(common)
    # calculatetext
    r_unique_pct = (len(r_unique) / total * 100) if total > 0 else 0
    nr_unique_pct = (len(nr_unique) / total * 100) if total > 0 else 0
    common_pct = (len(common) / total * 100) if total > 0 else 0
    # textresults
    print(f"\n{Type_level.upper()} Level Statistics:")
    print(f"-" * 60)
    print(f"R-onlymicrobetext: {len(r_unique)} ({r_unique_pct:.2f}%)")
    print(f"NR-onlymicrobetext: {len(nr_unique)} ({nr_unique_pct:.2f}%)")
    print(f"RandNRshared microbe count: {len(common)} ({common_pct:.2f}%)")
    print(f"textmicrobetext: {total}")
    print(f"-" * 60)
    # savestatisticsresults
    stats_df = pd.DataFrame({
        'Category': ['R_unique', 'NR_unique', 'Common', 'Total'],
        'Count': [len(r_unique), len(nr_unique), len(common), total],
        'Percentage': [f"{r_unique_pct:.2f}%", f"{nr_unique_pct:.2f}%",         f"{common_pct:.2f}%", "100.00%"]
    })
    stats_file = output_dir / f"CRC_{Type_level}_microbe_statistics.csv"
    stats_df.to_csv(stats_file, index=False)
    print(f"\nStatistics saved to: {stats_file}")
    # savetextmicrobetext
    microbe_lists = pd.DataFrame({
        'R_unique': list(r_unique) + [''] * (max(len(r_unique), len(nr_unique), len(common)) - len(r_unique)),
        'NR_unique': list(nr_unique) + [''] * (max(len(r_unique), len(nr_unique), len(common)) - len(nr_unique)),
        'Common': list(common) + [''] * (max(len(r_unique), len(nr_unique), len(common)) - len(common))
    })
    microbe_list_file = output_dir / f"CRC_{Type_level}_microbe_lists.csv"
    microbe_lists.to_csv(microbe_list_file, index=False)
    print(f"Microbe lists saved to: {microbe_list_file}")
    return stats_df

# text
if __name__ == "__main__":
    print("Starting CRC microbiome data processing...")
    print(f"Input directories:")
    for group, path in input_dirs.items():
        print(f" {group}: {path}")
        print(f"Output directory: {output_dir}")
        print(f"\nSample groups:")
        for group, samples in sample_groups.items():
            print(f" {group}: {', '.join(samples)}")
            # processgenusandspeciestext
            for Type_level in ['genus', 'species']:
                print(f"\n{'='*60}")
                print(f"Processing {Type_level.upper()} level")
                print(f"{'='*60}")
                # processRtextandNRtext
                for group_name, sample_list in sample_groups.items():
                    print(f"\nProcessing {group_name} group...")
                    get_matrix(Type_level, group_name, sample_list)
                    # calculatestatisticsdata
                    calculate_microbe_statistics(Type_level)
                    print("\n" + "="*60)
                    print("All processing completed!")
                    print("="*60)

### textrowstext

In [ ]:
import pandas as pd
import gzip
from pathlib import Path
from tqdm import tqdm

# textsampletext
sample_groups = {
    'NR': ['B2', 'B3', 'B6', 'C6'],
    'R': ['B1', 'B4', 'B7', 'E5']
}

# textinputoutputpath - text
input_dirs = {
    'NR': Path("/path/to/project/data_V3/special/CRC/NR/abundance_pathseq_V3"),
    'R': Path("/path/to/project/data_V3/special/CRC/R/abundance_pathseq_V3")
}

output_dir = Path("/path/to/project/results_V3/special/CRC/01.group_matrix")
output_dir.mkdir(parents=True, exist_ok=True)

def get_matrix(Type_level, group_name, sample_list):
    """
    processtextandtextdata
    Type_level: 'genus' or 'species'
    group_name: 'R' or 'NR'
    sample_list: sample columntext
    """
    old_eRPKM_df = pd.DataFrame()
    old_eRPKM_Normalized_df = pd.DataFrame()
    # textinput directory
    input_dir = input_dirs[group_name]
    for sample in tqdm(sample_list, desc=f"Processing {group_name} {Type_level}"):
        # filenameformat: B1_rna_output.pathseq.txt.gz or B1_rna_output.pathseq.txt
        file_path_gz = input_dir / f"{sample}_rna_output.pathseq.txt.gz"
        file_path_txt = input_dir / f"{sample}_rna_output.pathseq.txt"
        if file_path_gz.exists():
            file_path = file_path_gz
            open_func = lambda f: gzip.open(f, 'rt')
        elif file_path_txt.exists():
            file_path = file_path_txt
            open_func = lambda f: open(f, 'r')
        else:
            print(f"Warning: Neither {file_path_gz} nor {file_path_txt} found, skipping...")
            continue
        # readfile
        with open_func(file_path) as f:
            abundance = pd.read_csv(f, sep='\t')
            # texttypecolumnfiltergenusorspeciesdata
            abundance_filtered = abundance[abundance['type'] == Type_level]
            # extractscorecolumn
            eRPKM_df = abundance_filtered[["name", "score"]].copy()
            eRPKM_df.rename(columns={"score": sample}, inplace=True)
            # extractscore_normalizedcolumn
            eRPKM_Normalized_df = abundance_filtered[["name", "score_normalized"]].copy()
            eRPKM_Normalized_df.rename(columns={"score_normalized": sample}, inplace=True)
            # textdata
            if old_eRPKM_df.empty:
                old_eRPKM_df = eRPKM_df
            else:
                old_eRPKM_df = pd.merge(old_eRPKM_df, eRPKM_df, on="name", how="outer")
                if old_eRPKM_Normalized_df.empty:
                    old_eRPKM_Normalized_df = eRPKM_Normalized_df
                else:
                    old_eRPKM_Normalized_df = pd.merge(old_eRPKM_Normalized_df, eRPKM_Normalized_df, on="name", how="outer")
                    # text
                    old_eRPKM_df = old_eRPKM_df.fillna(0)
                    old_eRPKM_df.sort_values(by="name", inplace=True, ascending=True)
                    old_eRPKM_Normalized_df = old_eRPKM_Normalized_df.fillna(0)
                    old_eRPKM_Normalized_df.sort_values(by="name", inplace=True, ascending=True)
                    # savefile
                    output_file = output_dir / f"CRC_{group_name}_{Type_level}_pathseq.csv"
                    old_eRPKM_df.to_csv(output_file, sep=',', index=False)
                    print(f"Saved: {output_file}")
                    output_file_norm = output_dir / f"CRC_{group_name}_{Type_level}_pathseq_Normalized.csv"
                    old_eRPKM_Normalized_df.to_csv(output_file_norm, sep=',', index=False)
                    print(f"Saved: {output_file_norm}")
                    return old_eRPKM_df

def get_prevalent_microbes(df, prevalence_threshold=0.5):
    """
    textrowstextmicrobe
    df: microbetext, textcolumntextname, textcolumntextsample
    prevalence_threshold: textrowstext(text50%)
    return: textmicrobetext
    """
    # textsample column(textcolumnnametextallcolumn)
    sample_columns = df.columns[1:]
    n_samples = len(sample_columns)
    # calculatetextmicrobestextsamplesintext(score > 0)
    presence_matrix = df[sample_columns] > 0
    n_samples_present = presence_matrix.sum(axis=1)
    # calculatetextrowstext
    prevalence = n_samples_present / n_samples
    # filtertextmicrobe
    prevalent_microbes = set(df[prevalence >= prevalence_threshold]['name'])
    return prevalent_microbes, prevalence

def calculate_microbe_statistics(Type_level, prevalence_threshold=0.5):
    """
    calculateRtextandNR-onlytextsharedmicrobecountandtext
    textrowstext: onlystatisticstext>=50%sampleintextmicrobe
    Type_level: 'genus' or 'species'
    prevalence_threshold: textrowstext(text0.5text50%)
    """
    print(f"\n{'='*60}")
    print(f"Calculating statistics for {Type_level} (Prevalence >= {prevalence_threshold*100}%)")
    print(f"{'='*60}")
    # readRtextandNRtextdata
    r_file = output_dir / f"CRC_R_{Type_level}_pathseq.csv"
    nr_file = output_dir / f"CRC_NR_{Type_level}_pathseq.csv"
    r_df = pd.read_csv(r_file)
    nr_df = pd.read_csv(nr_file)
    # textsample count
    r_n_samples = len(r_df.columns) - 1 # subtract the name column
    nr_n_samples = len(nr_df.columns) - 1
    print(f"\nRtextsample count: {r_n_samples}")
    print(f"NRtextsample count: {nr_n_samples}")
    print(f"textrowstext: >= {prevalence_threshold*100}% sample")
    print(f"Rtext >= {int(r_n_samples * prevalence_threshold)} samplesintext")
    print(f"NRtext >= {int(nr_n_samples * prevalence_threshold)} samplesintext")
    # textrowstextmicrobe
    r_prevalent_microbes, r_prevalence = get_prevalent_microbes(r_df, prevalence_threshold)
    nr_prevalent_microbes, nr_prevalence = get_prevalent_microbes(nr_df, prevalence_threshold)
    print(f"\nRtextrowstextmicrobetext: {len(r_prevalent_microbes)}")
    print(f"NRtextrowstextmicrobetext: {len(nr_prevalent_microbes)}")
    # calculatetextwithandsharedmicrobe(textrowstextfilterafter)
    r_unique = r_prevalent_microbes - nr_prevalent_microbes # R-only
    nr_unique = nr_prevalent_microbes - r_prevalent_microbes # NR-only
    common = r_prevalent_microbes & nr_prevalent_microbes # shared
    # text
    total = len(r_unique) + len(nr_unique) + len(common)
    # calculatetext
    r_unique_pct = (len(r_unique) / total * 100) if total > 0 else 0
    nr_unique_pct = (len(nr_unique) / total * 100) if total > 0 else 0
    common_pct = (len(common) / total * 100) if total > 0 else 0
    # textresults
    print(f"\n{Type_level.upper()} Level Statistics (Prevalence-filtered):")
    print(f"-" * 60)
    print(f"R-onlymicrobetext: {len(r_unique)} ({r_unique_pct:.2f}%)")
    print(f"NR-onlymicrobetext: {len(nr_unique)} ({nr_unique_pct:.2f}%)")
    print(f"RandNRshared microbe count: {len(common)} ({common_pct:.2f}%)")
    print(f"textmicrobetext (prevalence-filtered): {total}")
    print(f"-" * 60)
    # savestatisticsresults
    stats_df = pd.DataFrame({
        'Category': ['R_unique', 'NR_unique', 'Common', 'Total'],
        'Count': [len(r_unique), len(nr_unique), len(common), total],
        'Percentage': [f"{r_unique_pct:.2f}%", f"{nr_unique_pct:.2f}%",         f"{common_pct:.2f}%", "100.00%"]
    })
    stats_file = output_dir / f"CRC_{Type_level}_microbe_statistics_prevalence{int(prevalence_threshold*100)}.csv"
    stats_df.to_csv(stats_file, index=False)
    print(f"\nStatistics saved to: {stats_file}")
    # savetextmicrobetext, textrowstextinformation
    max_len = max(len(r_unique), len(nr_unique), len(common))
    # forR-onlymicrobetextrowstextinformation
    r_unique_list = []
    for microbe in sorted(r_unique):
        prev = r_prevalence[r_df['name'] == microbe].values[0]
        r_unique_list.append(f"{microbe} ({prev*100:.1f}%)")
        r_unique_list += [''] * (max_len - len(r_unique))
        # forNR-onlymicrobetextrowstextinformation
        nr_unique_list = []
        for microbe in sorted(nr_unique):
            prev = nr_prevalence[nr_df['name'] == microbe].values[0]
            nr_unique_list.append(f"{microbe} ({prev*100:.1f}%)")
            nr_unique_list += [''] * (max_len - len(nr_unique))
            # forsharedmicrobetextrowstextinformation(RtextandNRtextrowstext)
            common_list = []
            for microbe in sorted(common):
                r_prev = r_prevalence[r_df['name'] == microbe].values[0]
                nr_prev = nr_prevalence[nr_df['name'] == microbe].values[0]
                common_list.append(f"{microbe} (R:{r_prev*100:.1f}%, NR:{nr_prev*100:.1f}%)")
                common_list += [''] * (max_len - len(common))
                microbe_lists = pd.DataFrame({
                    'R_unique': r_unique_list,
                    'NR_unique': nr_unique_list,
                    'Common': common_list
                })
                microbe_list_file = output_dir / f"CRC_{Type_level}_microbe_lists_prevalence{int(prevalence_threshold*100)}.csv"
                microbe_lists.to_csv(microbe_list_file, index=False)
                print(f"Microbe lists (with prevalence) saved to: {microbe_list_file}")
                # textsavedetailedtextrowstext
                all_microbes = sorted(r_prevalent_microbes.union(nr_prevalent_microbes))
                prevalence_detail = []
                for microbe in all_microbes:
                    r_prev = r_prevalence[r_df['name'] == microbe].values[0] if microbe in r_prevalent_microbes else 0
                    nr_prev = nr_prevalence[nr_df['name'] == microbe].values[0] if microbe in nr_prevalent_microbes else 0
                    if microbe in r_unique:
                        category = 'R_unique'
                    elif microbe in nr_unique:
                        category = 'NR_unique'
                    else:
                        category = 'Common'
                        prevalence_detail.append({
                            'Microbe': microbe,
                            'Category': category,
                            'R_prevalence': f"{r_prev*100:.1f}%",
                            'NR_prevalence': f"{nr_prev*100:.1f}%"
                        })
                        prevalence_detail_df = pd.DataFrame(prevalence_detail)
                        prevalence_detail_file = output_dir / f"CRC_{Type_level}_prevalence_detail.csv"
                        prevalence_detail_df.to_csv(prevalence_detail_file, index=False)
                        print(f"Prevalence detail saved to: {prevalence_detail_file}")
                        return stats_df

# text
if __name__ == "__main__":
    print("Starting CRC microbiome data processing...")
    print(f"Input directories:")
    for group, path in input_dirs.items():
        print(f" {group}: {path}")
        print(f"Output directory: {output_dir}")
        print(f"\nSample groups:")
        for group, samples in sample_groups.items():
            print(f" {group}: {', '.join(samples)}")
            # textrowstext
            PREVALENCE_THRESHOLD = 0.5 # 50%
            # processgenusandspeciestext
            for Type_level in ['genus', 'species']:
                print(f"\n{'='*60}")
                print(f"Processing {Type_level.upper()} level")
                print(f"{'='*60}")
                # processRtextandNRtext
                for group_name, sample_list in sample_groups.items():
                    print(f"\nProcessing {group_name} group...")
                    get_matrix(Type_level, group_name, sample_list)
                    # calculatestatisticsdata(textrowstext)
                    calculate_microbe_statistics(Type_level, prevalence_threshold=PREVALENCE_THRESHOLD)
                    print("\n" + "="*60)
                    print("All processing completed!")
                    print("="*60)

# microbegeneshare rate

## overallshare

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import os

# textfilepath
SAMPLE_INFO_PATH = "/path/to/project/data_V3/meta/cancer_status_sample_1897.csv"
BLAST_DIR = "/path/to/project/data_V3/genes/pathseq_reads_genes_public/public"

def read_sseqid_from_file(file_path):
    """fromblastresultsfileinreadallsseqid"""
    sseqids = set()
    try:
        if not os.path.exists(file_path):
            print(f"Warning: File not found: {file_path}")
            return sseqids
        with open(file_path, 'r') as f:
            next(f) # skip header
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) >= 3:
                    sseqid = parts[2] # sseqidtextthird column (index 2)
                    sseqids.add(sseqid)
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return sseqids

def collect_sseqids_by_cancer_status(sample_info_df, blast_dir):
    """textcancer typeandtextallsseqid"""
    cancer_status_sseqids = defaultdict(lambda: defaultdict(set))
    for idx, row in sample_info_df.iterrows():
        cancer_type = row['Cancer_Type']
        status = row['Status']
        sample = row['Sample']
        # buildblastfilepath
        blast_file = os.path.join(blast_dir, f"{sample}.diamond.blastx.filtered.txt")
        # readtextsamplesseqid
        sseqids = read_sseqid_from_file(blast_file)
        # texttotextcancer typeandtextin
        cancer_status_sseqids[cancer_type][status].update(sseqids)
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1} samples...")
            return cancer_status_sseqids

def calculate_statistics(normal_set, tumor_set):
    """calculateshared, textwithstatisticsdata"""
    shared = normal_set & tumor_set
    normal_only = normal_set - tumor_set
    tumor_only = tumor_set - normal_set
    # text normaltextwith + shared
    total = len(normal_only) + len(shared) + len(tumor_only)
    if total == 0:
        return {
        'shared_count': 0,
        'shared_pct': '0.0%',
        'normal_only_count': 0,
        'normal_only_pct': '0.0%',
        'tumor_only_count': len(tumor_only),
        'tumor_only_pct': 'N/A'
    }
    shared_pct = (len(shared) / total) * 100
    normal_only_pct = (len(normal_only) / total) * 100
    tumor_only_pct = (len(tumor_only) / total) * 100
    return {
        'shared_count': len(shared),
        'shared_pct': f"{shared_pct:.1f}%",
        'normal_only_count': len(normal_only),
        'normal_only_pct': f"{normal_only_pct:.1f}%",
        'tumor_only_count': len(tumor_only),
        'tumor_only_pct': f"{tumor_only_pct:.1f}%",
        'shared_set': shared,
        'normal_only_set': normal_only,
        'tumor_only_set': tumor_only
    }

def main():
    print("=== startstatisticscancer typemicrobegenedata ===\n")
    # 1. readsampleinformationtext
    print("1. readsampleinformationtext...")
    sample_info_df = pd.read_csv(SAMPLE_INFO_PATH)
    print(f"shared {len(sample_info_df)} samples\n")
    # 2. texteachcancer typeeachtextsseqid
    print("2. textsamplessseqid...")
    cancer_status_sseqids = collect_sseqids_by_cancer_status(sample_info_df, BLAST_DIR)
    print(f"textprocess {len(cancer_status_sseqids)} cancer type\n")
    # 3. textstatistics(text)
    print("=" * 80)
    print("textstatistics(text)")
    print("=" * 80)
    global_normal_set = set()
    global_tumor_set = set()
    for cancer_type, status_dict in cancer_status_sseqids.items():
        if 'Normal' in status_dict:
            global_normal_set.update(status_dict['Normal'])
            if 'Tumor' in status_dict:
                global_tumor_set.update(status_dict['Tumor'])
                global_stats = calculate_statistics(global_normal_set, global_tumor_set)
                print(f"\ntextNormal uniquesseqidtext: {global_stats['normal_only_count']:,} ({global_stats['normal_only_pct']})")
                print(f"textTumor uniquesseqidtext: {global_stats['tumor_only_count']:,} ({global_stats['tumor_only_pct']})")
                print(f"textsharedsseqidtext: {global_stats['shared_count']:,} ({global_stats['shared_pct']})")
                print(f"\ntextNormaltextsseqidtext: {len(global_normal_set):,}")
                print(f"textTumortextsseqidtext: {len(global_tumor_set):,}")
                # 4. eachcancer typestatistics
                print("\n" + "=" * 80)
                print("statistics by cancer type")
                print("=" * 80)
                results = []
                for cancer_type in sorted(cancer_status_sseqids.keys()):
                    status_dict = cancer_status_sseqids[cancer_type]
                    normal_set = status_dict.get('Normal', set())
                    tumor_set = status_dict.get('Tumor', set())
                    if len(normal_set) == 0 and len(tumor_set) == 0:
                        print(f"\n{cancer_type}: nowithtextdata")
                        continue
                    stats = calculate_statistics(normal_set, tumor_set)
                    print(f"\n{cancer_type}:")
                    print(f" Normalsamplesseqidtext: {len(normal_set):,}")
                    print(f" Tumorsamplesseqidtext: {len(tumor_set):,}")
                    print(f" sharedsseqid: {stats['shared_count']:,} ({stats['shared_pct']})")
                    print(f" Normal uniquesseqid: {stats['normal_only_count']:,} ({stats['normal_only_pct']})")
                    print(f" Tumor uniquesseqid: {stats['tumor_only_count']:,} ({stats['tumor_only_pct']})")
                    results.append({
                        'Cancer_type': cancer_type,
                        'Normal_total': len(normal_set),
                        'Tumor_total': len(tumor_set),
                        'Shared_count': stats['shared_count'],
                        'Shared_percentage': stats['shared_pct'],
                        'Normal_only_count': stats['normal_only_count'],
                        'Normal_only_percentage': stats['normal_only_pct'],
                        'Tumor_only_count': stats['tumor_only_count'],
                        'Tumor_only_percentage': stats['tumor_only_pct']
                    })
                    # 5. saveresultstoCSV
                    output_file = "/path/to/project/results_V3/summary/summary_genes/cancer_sseqid_statistics.csv"
                    results_df = pd.DataFrame(results)
                    results_df.to_csv(output_file, index=False)
                    print(f"\nresultssavedto: {output_file}")
                    # 6. savetextstatistics
                    global_output = "/path/to/project/results_V3/summary/summary_genes/global_sseqid_statistics.csv"
                    global_df = pd.DataFrame([{
                        'Type': 'Global',
                        'Normal_total': len(global_normal_set),
                        'Tumor_total': len(global_tumor_set),
                        'Shared_count': global_stats['shared_count'],
                        'Shared_percentage': global_stats['shared_pct'],
                        'Normal_only_count': global_stats['normal_only_count'],
                        'Normal_only_percentage': global_stats['normal_only_pct'],
                        'Tumor_only_count': global_stats['tumor_only_count'],
                        'Tumor_only_percentage': global_stats['tumor_only_pct']
                    }])
                    global_df.to_csv(global_output, index=False)
                    print(f"textstatisticssavedto: {global_output}")
                    print("\n=== statisticscompleted ===")

if __name__ == "__main__":
    main()

## sharing rate in common species

In [ ]:
import pandas as pd
import os
from collections import defaultdict
import re

# ==================== text ====================
taxid_file = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"
sample_info_file = "/path/to/project/data_V3/meta/cancer_status_sample_1897.csv"
microbe_dir = "/path/to/project/results_V3/cancers_V3.1"
blast_dir = "/path/to/project/data_V3/genes/pathseq_reads_genes_public/public"
output_dir = "/path/to/project/results_V3/summary/sharing_rate/gene_stable"

os.makedirs(output_dir, exist_ok=True)

cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

print("=" * 80)
print(" textmicrobetextgenetextanalyze")
print("=" * 80)
print("\nanalyzetext:")
print(" 1. textcancer typetextfilterNormal/Tumorintextmicrobe(≥50%sampleexists)")
print(" 2. textextractNormaltextmicrobeandTumortextmicrobetextgene")
print(" 3. text: Normalgenetext vs Tumorgenetext")
print("=" * 80)

# ==================== text1: buildmicrobetexttotaxidmapping ====================
print("\n[text1] loadmicrobetexttotaxidmapping...")

def normalize_name(name):
    return re.sub(r'[/\s]+', '_', str(name))

taxid_df = pd.read_csv(taxid_file, sep='\t')
microbe_to_taxids = defaultdict(list)
for _, row in taxid_df.iterrows():
    species_name = normalize_name(row['species_scientific_name'])
    microbe_to_taxids[species_name].append(str(row['tax_id']))

print(f" OK textload {len(microbe_to_taxids)} microbestotaxidmapping")

# ==================== text2: readsampleinformation ====================
print("\n[text2] loadsampleinformation...")

sample_info_df = pd.read_csv(sample_info_file)
sample_to_info = {}
for _, row in sample_info_df.iterrows():
    sample_to_info[row['Sample']] = {
        'cancer': row['Cancer_Type'],
        'status': row['Status']
    }

print(f" OK textload {len(sample_to_info)} samples")

# ==================== text3: filtertextcancer typetextmicrobetexttaxid ====================
print("\n[text3] filtertextcancer typeNormal/Tumortextmicrobe...")

def get_microbes_50pct_nonzero(file_path):
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = sample_data.shape[1] * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    return set(microbes)

def microbes_to_taxids(microbe_set):
    """textmicrobetextfortaxidtext"""
    taxid_set = set()
    for microbe in microbe_set:
        if microbe in microbe_to_taxids:
            taxid_set.update(microbe_to_taxids[microbe])
            return taxid_set

# textcancer typeNormalandTumortextmicrobetexttaxid
cancer_taxids = {
    'Normal': {}, # cancer -> taxid_set
    'Tumor': {} # cancer -> taxid_set
}

base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_species_pathseq.csv"

for cancer in cancer_types:
    try:
        # readNormalandTumortextmicrobe
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")
        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)
        # textfortaxid
        cancer_taxids['Normal'][cancer] = microbes_to_taxids(normal_microbes)
        cancer_taxids['Tumor'][cancer] = microbes_to_taxids(tumor_microbes)
        print(f" {cancer}:")
        print(f" Normal: {len(normal_microbes)} microbe → {len(cancer_taxids['Normal'][cancer])} taxid")
        print(f" Tumor: {len(tumor_microbes)} microbe → {len(cancer_taxids['Tumor'][cancer])} taxid")
    except Exception as e:
        print(f" Warning️ {cancer} processfailed: {e}")

# ==================== text4: fromblastfileextractgene ====================
print("\n[text4] extracttextmicrobetextgene...")

def extract_genes_from_blast(blast_file, target_taxids):
    genes = set()
    if not os.path.exists(blast_file):
        return genes
    try:
        with open(blast_file, 'r') as f:
            next(f) # skip header
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) < 3:
                    continue
                qseqid = parts[0]
                sseqid = parts[2]
                if 'CTAXA:' in qseqid:
                    ctaxa_match = re.search(r'CTAXA:([0-9,]+)', qseqid)
                    if ctaxa_match:
                        taxids_in_record = ctaxa_match.group(1).split(',')
                        if any(tid in target_taxids for tid in taxids_in_record):
                            if sseqid:
                                genes.add(sseqid)
    except Exception as e:
        pass
    return genes

# ==================== text5: textcancer typeandtextgene ====================
print("\n[text5] fromtextcancer typesampleinextractgene...")

# datatext: status -> cancer -> gene_set
cancer_genes = {
    'Normal': defaultdict(set),
    'Tumor': defaultdict(set)
}

# text: status -> gene_set
pan_cancer_genes = {
    'Normal': set(),
    'Tumor': set()
}

processed = 0
for sample, info in sample_to_info.items():
    cancer = info['cancer']
    status = info['status']
    if cancer not in cancer_types:
        continue
    blast_file = os.path.join(blast_dir, f"{sample}.diamond.blastx.filtered.txt")
    # textsampletext, extracttextmicrobegene
    target_taxids = cancer_taxids[status].get(cancer, set())
    if target_taxids:
        genes = extract_genes_from_blast(blast_file, target_taxids)
        cancer_genes[status][cancer].update(genes)
        processed += 1
        if processed % 100 == 0:
            print(f" processed {processed}/{len(sample_to_info)} samples...")

# ==================== text6: text ====================
print("\n[text6] textgene...")

for status in ['Normal', 'Tumor']:
    for cancer in cancer_types:
        pan_cancer_genes[status].update(cancer_genes[status][cancer])

print(f" Normalgenetext: {len(pan_cancer_genes['Normal'])} genes")
print(f" Tumorgenetext: {len(pan_cancer_genes['Tumor'])} genes")

# ==================== text7: statisticstext ====================
print("\n[text7] statisticsgenetext...")

results = []

# statistics by cancer type
for cancer in cancer_types:
    normal_genes = cancer_genes['Normal'][cancer]
    tumor_genes = cancer_genes['Tumor'][cancer]
    intersection = normal_genes & tumor_genes
    union = normal_genes | tumor_genes
    normal_unique = normal_genes - tumor_genes
    tumor_unique = tumor_genes - normal_genes
    if len(union) > 0:
        results.append({
            'Cancer': cancer,
            'Normal_genes': len(normal_genes),
            'Tumor_genes': len(tumor_genes),
            'Shared_genes': len(intersection),
            'Shared_pct': f"{len(intersection) / len(union) * 100:.1f}%",
            'Normal_unique': len(normal_unique),
            'Normal_unique_pct': f"{len(normal_unique) / len(union) * 100:.1f}%",
            'Tumor_unique': len(tumor_unique),
            'Tumor_unique_pct': f"{len(tumor_unique) / len(union) * 100:.1f}%"
        })
        print(f"\n {cancer}:")
        print(f" Normalgene: {len(normal_genes):,}")
        print(f" Tumorgene: {len(tumor_genes):,}")
        print(f" Normal uniquegene: {len(normal_unique):,} ({len(normal_unique)/len(union)*100:.1f}%)")
        print(f" Tumor uniquegene: {len(tumor_unique):,} ({len(tumor_unique)/len(union)*100:.1f}%)")
        print(f" textgene: {len(intersection):,} ({len(intersection)/len(union)*100:.1f}%)")

# textstatistics
pan_intersection = pan_cancer_genes['Normal'] & pan_cancer_genes['Tumor']
pan_union = pan_cancer_genes['Normal'] | pan_cancer_genes['Tumor']
pan_normal_unique = pan_cancer_genes['Normal'] - pan_cancer_genes['Tumor']
pan_tumor_unique = pan_cancer_genes['Tumor'] - pan_cancer_genes['Normal']

results.append({
    'Cancer': 'PanCancer',
    'Normal_genes': len(pan_cancer_genes['Normal']),
    'Tumor_genes': len(pan_cancer_genes['Tumor']),
    'Shared_genes': len(pan_intersection),
    'Shared_pct': f"{len(pan_intersection) / len(pan_union) * 100:.1f}%",
    'Normal_unique': len(pan_normal_unique),
    'Normal_unique_pct': f"{len(pan_normal_unique) / len(pan_union) * 100:.1f}%",
    'Tumor_unique': len(pan_tumor_unique),
    'Tumor_unique_pct': f"{len(pan_tumor_unique) / len(pan_union) * 100:.1f}%"
})

print(f"\n text:")
print(f" Normalgenetext: {len(pan_cancer_genes['Normal']):,}")
print(f" Tumorgenetext: {len(pan_cancer_genes['Tumor']):,}")
print(f" Normal uniquegene: {len(pan_normal_unique):,} ({len(pan_normal_unique)/len(pan_union)*100:.1f}%)")
print(f" Tumor uniquegene: {len(pan_tumor_unique):,} ({len(pan_tumor_unique)/len(pan_union)*100:.1f}%)")
print(f" textgene: {len(pan_intersection):,} ({len(pan_intersection)/len(pan_union)*100:.1f}%)")

# ==================== saveresults ====================
output_file = os.path.join(output_dir, "stable_microbes_gene_sharing.csv")
df_results = pd.DataFrame(results)
df_results.to_csv(output_file, index=False)

print(f"\nOK resultssavedto: {output_file}")
print("\n" + "=" * 80)
print("analyzecompleted!")
print("=" * 80)



In [ ]:
import pandas as pd
import os
from collections import defaultdict
import re

# ==================== text ====================
taxid_file = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"
sample_info_file = "/path/to/project/data_V3/meta/cancer_status_sample_1897.csv"
microbe_dir = "/path/to/project/results_V3/cancers_V3.1"
blast_dir = "/path/to/project/data_V3/genes/pathseq_reads_genes_public/public"
output_dir = "/path/to/project/results_V3/summary/sharing_rate_gene"

os.makedirs(output_dir, exist_ok=True)

cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

print("=" * 80)
print(" textmicrobetextgenetextanalyze")
print("=" * 80)
print("\nanalyzetext:")
print(" 1. textcancer typetextfilterNormal/Tumorintextmicrobe(≥50%sampleexists)")
print(" 2. textextractNormaltextmicrobeandTumortextmicrobetextgene")
print(" 3. text: Normalgenetext vs Tumorgenetext")
print("=" * 80)

# ==================== text1: buildmicrobetexttotaxidmapping ====================
print("\n[text1] loadmicrobetexttotaxidmapping...")

def normalize_name(name):
    return re.sub(r'[/\s]+', '_', str(name))

taxid_df = pd.read_csv(taxid_file, sep='\t')
microbe_to_taxids = defaultdict(list)
for _, row in taxid_df.iterrows():
    species_name = normalize_name(row['species_scientific_name'])
    microbe_to_taxids[species_name].append(str(row['tax_id']))

print(f" OK textload {len(microbe_to_taxids)} microbestotaxidmapping")

# ==================== text2: readsampleinformation ====================
print("\n[text2] loadsampleinformation...")

sample_info_df = pd.read_csv(sample_info_file)
sample_to_info = {}
for _, row in sample_info_df.iterrows():
    sample_to_info[row['Sample']] = {
        'cancer': row['Cancer_Type'],
        'status': row['Status']
    }

print(f" OK textload {len(sample_to_info)} samples")

# ==================== text3: filtertextcancer typetextmicrobetexttaxid ====================
print("\n[text3] filtertextcancer typeNormal/Tumortextmicrobe...")

def get_microbes_50pct_nonzero(file_path):
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = sample_data.shape[1] * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    return set(microbes)

def microbes_to_taxids(microbe_set):
    """textmicrobetextfortaxidtext"""
    taxid_set = set()
    for microbe in microbe_set:
        if microbe in microbe_to_taxids:
            taxid_set.update(microbe_to_taxids[microbe])
            return taxid_set

# textcancer typeNormalandTumortextmicrobetexttaxid
cancer_taxids = {
    'Normal': {}, # cancer -> taxid_set
    'Tumor': {} # cancer -> taxid_set
}

base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_species_pathseq.csv"

for cancer in cancer_types:
    try:
        # readNormalandTumortextmicrobe
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")
        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)
        # textfortaxid
        cancer_taxids['Normal'][cancer] = microbes_to_taxids(normal_microbes)
        cancer_taxids['Tumor'][cancer] = microbes_to_taxids(tumor_microbes)
        print(f" {cancer}:")
        print(f" Normal: {len(normal_microbes)} microbe → {len(cancer_taxids['Normal'][cancer])} taxid")
        print(f" Tumor: {len(tumor_microbes)} microbe → {len(cancer_taxids['Tumor'][cancer])} taxid")
    except Exception as e:
        print(f" Warning️ {cancer} processfailed: {e}")

# ==================== text4: fromblastfileextractgene ====================
print("\n[text4] extracttextmicrobetextgene...")

def extract_genes_from_blast(blast_file, target_taxids):
    genes = set()
    if not os.path.exists(blast_file):
        return genes
    try:
        with open(blast_file, 'r') as f:
            next(f) # skip header
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) < 3:
                    continue
                qseqid = parts[0]
                sseqid = parts[2]
                if 'CTAXA:' in qseqid:
                    ctaxa_match = re.search(r'CTAXA:([0-9,]+)', qseqid)
                    if ctaxa_match:
                        taxids_in_record = ctaxa_match.group(1).split(',')
                        if any(tid in target_taxids for tid in taxids_in_record):
                            if sseqid:
                                genes.add(sseqid)
    except Exception as e:
        pass
    return genes

# ==================== text5: textcancer typeandtextgene ====================
print("\n[text5] fromtextcancer typesampleinextractgene...")

# datatext: status -> cancer -> gene_set
cancer_genes = {
    'Normal': defaultdict(set),
    'Tumor': defaultdict(set)
}

# text: status -> gene_set
pan_cancer_genes = {
    'Normal': set(),
    'Tumor': set()
}

processed = 0
for sample, info in sample_to_info.items():
    cancer = info['cancer']
    status = info['status']
    if cancer not in cancer_types:
        continue
    blast_file = os.path.join(blast_dir, f"{sample}.diamond.blastx.filtered.txt")
    # textsampletext, extracttextmicrobegene
    target_taxids = cancer_taxids[status].get(cancer, set())
    if target_taxids:
        genes = extract_genes_from_blast(blast_file, target_taxids)
        cancer_genes[status][cancer].update(genes)
        processed += 1
        if processed % 100 == 0:
            print(f" processed {processed}/{len(sample_to_info)} samples...")

# ==================== text6: text ====================
print("\n[text6] textgene...")

for status in ['Normal', 'Tumor']:
    for cancer in cancer_types:
        pan_cancer_genes[status].update(cancer_genes[status][cancer])

print(f" Normalgenetext: {len(pan_cancer_genes['Normal'])} genes")
print(f" Tumorgenetext: {len(pan_cancer_genes['Tumor'])} genes")

# ==================== text7: statisticstextsavegenelist ====================
print("\n[text7] statisticsgenetextsavegenelist...")

results = []

def save_gene_lists(cancer_name, normal_genes, tumor_genes, intersection, normal_unique, tumor_unique):
    """savetextcancer typeallgenelist"""
    cancer_dir = os.path.join(output_dir, cancer_name)
    os.makedirs(cancer_dir, exist_ok=True)
    # textfortextafterlisttext
    gene_lists = {
        'Normal_genes.txt': sorted(normal_genes),
        'Tumor_genes.txt': sorted(tumor_genes),
        'Shared_genes.txt': sorted(intersection),
        'Normal_unique_genes.txt': sorted(normal_unique),
        'Tumor_unique_genes.txt': sorted(tumor_unique)
    }
    for filename, gene_list in gene_lists.items():
        filepath = os.path.join(cancer_dir, filename)
        with open(filepath, 'w') as f:
            f.write('\n'.join(gene_list))
            return cancer_dir

# statistics by cancer typetextsave
for cancer in cancer_types:
    normal_genes = cancer_genes['Normal'][cancer]
    tumor_genes = cancer_genes['Tumor'][cancer]
    intersection = normal_genes & tumor_genes
    union = normal_genes | tumor_genes
    normal_unique = normal_genes - tumor_genes
    tumor_unique = tumor_genes - normal_genes
    if len(union) > 0:
        # savegenelist
        saved_dir = save_gene_lists(cancer, normal_genes, tumor_genes,             intersection, normal_unique, tumor_unique)
        results.append({
            'Cancer': cancer,
            'Normal_genes': len(normal_genes),
            'Tumor_genes': len(tumor_genes),
            'Shared_genes': len(intersection),
            'Shared_pct': f"{len(intersection) / len(union) * 100:.1f}%",
            'Normal_unique': len(normal_unique),
            'Normal_unique_pct': f"{len(normal_unique) / len(union) * 100:.1f}%",
            'Tumor_unique': len(tumor_unique),
            'Tumor_unique_pct': f"{len(tumor_unique) / len(union) * 100:.1f}%"
        })
        print(f"\n {cancer}:")
        print(f" Normalgene: {len(normal_genes):,}")
        print(f" Tumorgene: {len(tumor_genes):,}")
        print(f" Normal uniquegene: {len(normal_unique):,} ({len(normal_unique)/len(union)*100:.1f}%)")
        print(f" Tumor uniquegene: {len(tumor_unique):,} ({len(tumor_unique)/len(union)*100:.1f}%)")
        print(f" textgene: {len(intersection):,} ({len(intersection)/len(union)*100:.1f}%)")
        print(f" OK genelistsavedto: {saved_dir}")

# textstatisticstextsave
pan_intersection = pan_cancer_genes['Normal'] & pan_cancer_genes['Tumor']
pan_union = pan_cancer_genes['Normal'] | pan_cancer_genes['Tumor']
pan_normal_unique = pan_cancer_genes['Normal'] - pan_cancer_genes['Tumor']
pan_tumor_unique = pan_cancer_genes['Tumor'] - pan_cancer_genes['Normal']

# savetextgenelist
pan_saved_dir = save_gene_lists('PanCancer', pan_cancer_genes['Normal'],     pan_cancer_genes['Tumor'], pan_intersection,     pan_normal_unique, pan_tumor_unique)

results.append({
    'Cancer': 'PanCancer',
    'Normal_genes': len(pan_cancer_genes['Normal']),
    'Tumor_genes': len(pan_cancer_genes['Tumor']),
    'Shared_genes': len(pan_intersection),
    'Shared_pct': f"{len(pan_intersection) / len(pan_union) * 100:.1f}%",
    'Normal_unique': len(pan_normal_unique),
    'Normal_unique_pct': f"{len(pan_normal_unique) / len(pan_union) * 100:.1f}%",
    'Tumor_unique': len(pan_tumor_unique),
    'Tumor_unique_pct': f"{len(pan_tumor_unique) / len(pan_union) * 100:.1f}%"
})

print(f"\n text:")
print(f" Normalgenetext: {len(pan_cancer_genes['Normal']):,}")
print(f" Tumorgenetext: {len(pan_cancer_genes['Tumor']):,}")
print(f" Normal uniquegene: {len(pan_normal_unique):,} ({len(pan_normal_unique)/len(pan_union)*100:.1f}%)")
print(f" Tumor uniquegene: {len(pan_tumor_unique):,} ({len(pan_tumor_unique)/len(pan_union)*100:.1f}%)")
print(f" textgene: {len(pan_intersection):,} ({len(pan_intersection)/len(pan_union)*100:.1f}%)")
print(f" OK genelistsavedto: {pan_saved_dir}")

# ==================== savetextresults ====================
output_file = os.path.join(output_dir, "stable_microbes_gene_sharing_summary.csv")
df_results = pd.DataFrame(results)
df_results.to_csv(output_file, index=False)

print(f"\nOK textresultssavedto: {output_file}")
print("\n" + "=" * 80)
print("filetext:")
print(" eachcancer type/textwithtextfiletext, text5files:")
print(" - Normal_genes.txt: Normalgenetext")
print(" - Tumor_genes.txt: Tumorgenetext")
print(" - Shared_genes.txt: textgene")
print(" - Normal_unique_genes.txt: Normal uniquegene")
print(" - Tumor_unique_genes.txt: Tumor uniquegene")
print("=" * 80)
print("analyzecompleted!")
print("=" * 80)

In [ ]:
import pandas as pd
import os

# allcancer typetext
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# filepathtext
base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_species_pathseq.csv"

# output directory
output_dir = "/path/to/project/results_V3/summary/sharing_rate_species"
os.makedirs(output_dir, exist_ok=True)

def get_microbes_50pct_nonzero(file_path):
    """readcsvfiletextreturntext50%textrowstextmicrobetext"""
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = sample_data.shape[1] * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    return set(microbes)

def save_microbe_lists(cancer_name, normal_microbes, tumor_microbes):
    """savetextcancer typeallmicrobe list(5files)"""
    cancer_dir = os.path.join(output_dir, cancer_name)
    os.makedirs(cancer_dir, exist_ok=True)
    # calculatetext
    intersection = normal_microbes & tumor_microbes
    normal_unique = normal_microbes - tumor_microbes
    tumor_unique = tumor_microbes - normal_microbes
    # textfortextafterlist
    microbe_lists = {
        'Normal_microbes.txt': sorted(normal_microbes),
        'Tumor_microbes.txt': sorted(tumor_microbes),
        'Shared_microbes.txt': sorted(intersection),
        'Normal_unique_microbes.txt': sorted(normal_unique),
        'Tumor_unique_microbes.txt': sorted(tumor_unique)
    }
    # saveallfile
    for filename, microbe_list in microbe_lists.items():
        filepath = os.path.join(cancer_dir, filename)
        with open(filepath, 'w') as f:
            f.write('\n'.join(microbe_list))
            return cancer_dir, len(intersection), len(normal_unique), len(tumor_unique)

print("=" * 80)
print(" textcancer typemicrobetextanalyze(5fileoutput)")
print("=" * 80)

# ========== text: text ==========
print("\n[text1] calculatetextmicrobe...")

# textallcancer type normal and tumor microbe
all_normal_microbes = set()
all_tumor_microbes = set()

for cancer in cancer_types:
    try:
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")
        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)
        all_normal_microbes.update(normal_microbes)
        all_tumor_microbes.update(tumor_microbes)
        print(f" {cancer}: Normal={len(normal_microbes)}, Tumor={len(tumor_microbes)}")
    except Exception as e:
        print(f" Warning️ process {cancer} text: {e}")

# savetext5files
pan_saved_dir, pan_shared, pan_normal_unique, pan_tumor_unique = save_microbe_lists(
    'PanCancer', all_normal_microbes, all_tumor_microbes
)

pan_union = all_normal_microbes | all_tumor_microbes

print(f"\n text:")
print(f" Normalmicrobetext: {len(all_normal_microbes):,}")
print(f" Tumormicrobetext: {len(all_tumor_microbes):,}")
print(f" Normal unique: {pan_normal_unique:,} ({pan_normal_unique/len(pan_union)*100:.1f}%)")
print(f" Tumor unique: {pan_tumor_unique:,} ({pan_tumor_unique/len(pan_union)*100:.1f}%)")
print(f" sharedmicrobe: {pan_shared:,} ({pan_shared/len(pan_union)*100:.1f}%)")
print(f" OK saved5filesto: {pan_saved_dir}")

# ========== text: textcancer typetext ==========
print("\n[text2] calculatetextcancer typetextmicrobe...")

results = []

for cancer in cancer_types:
    try:
        # buildpath
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")
        # readdata
        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)
        # save5files
        saved_dir, shared_count, normal_unique_count, tumor_unique_count = save_microbe_lists(
            cancer, normal_microbes, tumor_microbes
)
# calculatestatistics
union = normal_microbes | tumor_microbes
print(f"\n {cancer}:")
print(f" Normalmicrobe: {len(normal_microbes):,}")
print(f" Tumormicrobe: {len(tumor_microbes):,}")
print(f" Normal unique: {normal_unique_count:,} ({normal_unique_count/len(union)*100:.1f}%)")
print(f" Tumor unique: {tumor_unique_count:,} ({tumor_unique_count/len(union)*100:.1f}%)")
print(f" sharedmicrobe: {shared_count:,} ({shared_count/len(union)*100:.1f}%)")
print(f" OK saved5filesto: {saved_dir}")
# textresultstext
results.append({
    'Cancer': cancer,
    'Normal_microbes': len(normal_microbes),
    'Tumor_microbes': len(tumor_microbes),
    'Shared_microbes': shared_count,
    'Shared_pct': f"{shared_count / len(union) * 100:.1f}%",
    'Normal_unique': normal_unique_count,
    'Normal_unique_pct': f"{normal_unique_count / len(union) * 100:.1f}%",
    'Tumor_unique': tumor_unique_count,
    'Tumor_unique_pct': f"{tumor_unique_count / len(union) * 100:.1f}%"
})
except Exception as e:
    print(f" Warning️ process {cancer} text: {e}")

# textresultstotext
results.append({
    'Cancer': 'PanCancer',
    'Normal_microbes': len(all_normal_microbes),
    'Tumor_microbes': len(all_tumor_microbes),
    'Shared_microbes': pan_shared,
    'Shared_pct': f"{pan_shared / len(pan_union) * 100:.1f}%",
    'Normal_unique': pan_normal_unique,
    'Normal_unique_pct': f"{pan_normal_unique / len(pan_union) * 100:.1f}%",
    'Tumor_unique': pan_tumor_unique,
    'Tumor_unique_pct': f"{pan_tumor_unique / len(pan_union) * 100:.1f}%"
})

# ========== savetextresults ==========
output_file = os.path.join(output_dir, "microbe_sharing_summary.csv")
df_results = pd.DataFrame(results)
df_results.to_csv(output_file, index=False)

print(f"\nOK textresultssavedto: {output_file}")

print("\n" + "=" * 80)
print("filetext:")
print(" eachcancer type/textwithtextfiletext, text5files:")
print(" - Normal_microbes.txt: Normalmicrobetext")
print(" - Tumor_microbes.txt: Tumormicrobetext")
print(" - Shared_microbes.txt: textmicrobe")
print(" - Normal_unique_microbes.txt: Normal uniquemicrobe")
print(" - Tumor_unique_microbes.txt: Tumor uniquemicrobe")
print("=" * 80)
print(f"generated total {len(cancer_types) + 1} filestext, textfilestext5txtfile")
print("=" * 80)

# microbepeptide share rate

## overallshare

In [ ]:
import pandas as pd
import gzip
import os
from pathlib import Path
from collections import defaultdict

def read_peptides_from_file(file_path):
    """readpeptidedata files, returnpeptidetext"""
    peptides = set()
    try:
        if file_path.endswith('.gz'):
            with gzip.open(file_path, 'rt') as f:
                df = pd.read_csv(f, sep='\t')
        else:
            df = pd.read_csv(file_path, sep='\t')
            if 'peptide' in df.columns:
                peptides = set(df['peptide'].dropna().unique())
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return peptides

def get_sample_peptides(sample_name, hlai_dir, hlaii_dir):
    """textsamplesallpeptides(HLAIandHLAIItext)"""
    all_peptides = set()
    # readHLAIdata
    hlai_file = Path(hlai_dir) / f"{sample_name}.pHLA_I.tsv.gz"
    if hlai_file.exists():
        all_peptides.update(read_peptides_from_file(str(hlai_file)))
        # readHLAIIdata
        hlaii_file = Path(hlaii_dir) / f"{sample_name}.pHLA_II.tsv.gz"
        if hlaii_file.exists():
            all_peptides.update(read_peptides_from_file(str(hlaii_file)))
            return all_peptides

def save_peptide_lists(cancer_name, normal_peptides, tumor_peptides, output_base_dir):
    """savetextcancer typeallpeptidelist"""
    cancer_dir = os.path.join(output_base_dir, cancer_name)
    os.makedirs(cancer_dir, exist_ok=True)
    # calculatetextpeptides
    shared_peptides = normal_peptides & tumor_peptides
    normal_unique = normal_peptides - tumor_peptides
    tumor_unique = tumor_peptides - normal_peptides
    # textfortextafterlist
    peptide_lists = {
        'Normal_peptides.txt': sorted(normal_peptides),
        'Tumor_peptides.txt': sorted(tumor_peptides),
        'Shared_peptides.txt': sorted(shared_peptides),
        'Normal_unique_peptides.txt': sorted(normal_unique),
        'Tumor_unique_peptides.txt': sorted(tumor_unique)
    }
    # savefile
    for filename, peptide_list in peptide_lists.items():
        filepath = os.path.join(cancer_dir, filename)
        with open(filepath, 'w') as f:
            f.write('\n'.join(peptide_list))
            return cancer_dir, shared_peptides, normal_unique, tumor_unique

def main():
    # Path configuration
    data_csv = "/path/to/project/data_V3/meta/cancer_status_sample_1897.csv"
    hlai_dir = "/path/to/project/data_V3/peptides/pathseq_reads_peptides_public/filtered_out/BindLevel/HLAI"
    hlaii_dir = "/path/to/project/data_V3/peptides/pathseq_reads_peptides_public/filtered_out/BindLevel/HLAII"
    output_base_dir = "/path/to/project/results_V3/summary/shareing_rate_peptide/public/without_speciesINFO"
    # Create output directory
    os.makedirs(output_base_dir, exist_ok=True)
    # readsampleinformation
    print("=" * 80)
    print(" Peptidetextanalyze")
    print("=" * 80)
    print("\nanalyzetext:")
    print(" 1. textcancer typetextNormal/Tumorallsamplepeptides")
    print(" 2. text: Normal peptidetext vs Tumor peptidetext")
    print("=" * 80)
    print("\n[text1] readsampleinformation...")
    df_samples = pd.read_csv(data_csv)
    # textcancer typetext
    cancer_types = sorted(df_samples['Cancer_Type'].unique())
    print(f" OK textto {len(cancer_types)} cancer type")
    # text(text)
    global_normal_peptides = set()
    global_tumor_peptides = set()
    # texteachcancer typepeptides(textaftertextsave)
    cancer_peptides = {
        'Normal': {},
        'Tumor': {}
    }
    # textresults
    results = []
    print("\n[text2] processtextcancer typedatatextsavepeptidelist...")
    for cancer_type in cancer_types:
        print(f"\n {cancer_type}:")
        # textcancer typenormalandtumorsample
        cancer_df = df_samples[df_samples['Cancer_Type'] == cancer_type]
        normal_samples = cancer_df[cancer_df['Status'] == 'Normal']['Sample'].tolist()
        tumor_samples = cancer_df[cancer_df['Status'] == 'Tumor']['Sample'].tolist()
        print(f" sample count - Normal: {len(normal_samples)}, Tumor: {len(tumor_samples)}")
        # textcancer typeallnormalsamplepeptides
        cancer_normal_peptides = set()
        for i, sample in enumerate(normal_samples, 1):
            peptides = get_sample_peptides(sample, hlai_dir, hlaii_dir)
            cancer_normal_peptides.update(peptides)
            global_normal_peptides.update(peptides)
            if i % 20 == 0:
                print(f" processNormalsample: {i}/{len(normal_samples)}", end='\r')
                if normal_samples:
                    print(f" processNormalsample: {len(normal_samples)}/{len(normal_samples)} OK")
                    # textcancer typealltumorsamplepeptides
                    cancer_tumor_peptides = set()
                    for i, sample in enumerate(tumor_samples, 1):
                        peptides = get_sample_peptides(sample, hlai_dir, hlaii_dir)
                        cancer_tumor_peptides.update(peptides)
                        global_tumor_peptides.update(peptides)
                        if i % 20 == 0:
                            print(f" processTumorsample: {i}/{len(tumor_samples)}", end='\r')
                            if tumor_samples:
                                print(f" processTumorsample: {len(tumor_samples)}/{len(tumor_samples)} OK")
                                # savetextcancer typepeptides
                                cancer_peptides['Normal'][cancer_type] = cancer_normal_peptides
                                cancer_peptides['Tumor'][cancer_type] = cancer_tumor_peptides
                                # savepeptidelist
                                saved_dir, shared, normal_unique, tumor_unique = save_peptide_lists(
                                    cancer_type, cancer_normal_peptides, cancer_tumor_peptides, output_base_dir
)
# calculatestatistics
total_count = len(normal_unique) + len(shared) + len(tumor_unique)
if total_count > 0:
    shared_pct = (len(shared) / total_count) * 100
    normal_only_pct = (len(normal_unique) / total_count) * 100
    tumor_only_pct = (len(tumor_unique) / total_count) * 100
else:
    shared_pct = normal_only_pct = tumor_only_pct = 0.0
    # saveresults
    results.append({
        'Cancer_type': cancer_type,
        'Normal_samples': len(normal_samples),
        'Tumor_samples': len(tumor_samples),
        'Total_normal_peptides': len(cancer_normal_peptides),
        'Total_tumor_peptides': len(cancer_tumor_peptides),
        'Shared_peptides': len(shared),
        'Shared_percentage': f"{shared_pct:.1f}%",
        'Normal_unique_peptides': len(normal_unique),
        'Normal_unique_percentage': f"{normal_only_pct:.1f}%",
        'Tumor_unique_peptides': len(tumor_unique),
        'Tumor_unique_percentage': f"{tumor_only_pct:.1f}%"
    })
    print(f" Normal peptides: {len(cancer_normal_peptides):,}")
    print(f" Tumor peptides: {len(cancer_tumor_peptides):,}")
    print(f" Normal unique: {len(normal_unique):,} ({normal_only_pct:.1f}%)")
    print(f" Tumor unique: {len(tumor_unique):,} ({tumor_only_pct:.1f}%)")
    print(f" textpeptides: {len(shared):,} ({shared_pct:.1f}%)")
    print(f" OK Peptidelistsavedto: {saved_dir}")
    # calculatetextstatisticstextsave
    print("\n" + "=" * 80)
    print("[text3] textstatistics...")
    # savetextpeptidelist
    pan_saved_dir, global_shared, global_normal_only, global_tumor_only = save_peptide_lists(
        'PanCancer', global_normal_peptides, global_tumor_peptides, output_base_dir
)
global_total = len(global_normal_only) + len(global_shared) + len(global_tumor_only)
if global_total > 0:
    global_shared_pct = (len(global_shared) / global_total) * 100
    global_normal_only_pct = (len(global_normal_only) / global_total) * 100
    global_tumor_only_pct = (len(global_tumor_only) / global_total) * 100
else:
    global_shared_pct = global_normal_only_pct = global_tumor_only_pct = 0.0
    print(f"\n text:")
    print(f" Normal peptidetext: {len(global_normal_peptides):,}")
    print(f" Tumor peptidetext: {len(global_tumor_peptides):,}")
    print(f" Normal unique: {len(global_normal_only):,} ({global_normal_only_pct:.1f}%)")
    print(f" Tumor unique: {len(global_tumor_only):,} ({global_tumor_only_pct:.1f}%)")
    print(f" textpeptides: {len(global_shared):,} ({global_shared_pct:.1f}%)")
    print(f" OK Peptidelistsavedto: {pan_saved_dir}")
    # savestatistics by cancer typeresultstoCSV
    df_results = pd.DataFrame(results)
    output_file = os.path.join(output_base_dir, "peptide_comparison_statistics.csv")
    df_results.to_csv(output_file, index=False)
    print(f"\nOK statistics by cancer typeresultssavedto: {output_file}")
    # savetextstatistics
    global_stats = pd.DataFrame([{
        'Category': 'PanCancer',
        'Total_Normal_peptides': len(global_normal_peptides),
        'Total_Tumor_peptides': len(global_tumor_peptides),
        'Shared_peptides': len(global_shared),
        'Shared_percentage': f"{global_shared_pct:.1f}%",
        'Normal_unique_peptides': len(global_normal_only),
        'Normal_unique_percentage': f"{global_normal_only_pct:.1f}%",
        'Tumor_unique_peptides': len(global_tumor_only),
        'Tumor_unique_percentage': f"{global_tumor_only_pct:.1f}%"
    }])
    global_output = os.path.join(output_base_dir, "peptide_pan_cancer_statistics.csv")
    global_stats.to_csv(global_output, index=False)
    print(f"OK textstatisticsresultssavedto: {global_output}")
    print("\n" + "=" * 80)
    print("filetext:")
    print(" eachcancer type/textwithtextfiletext, text5files:")
    print(" - Normal_peptides.txt: Normal peptidetext")
    print(" - Tumor_peptides.txt: Tumor peptidetext")
    print(" - Shared_peptides.txt: textpeptides")
    print(" - Normal_unique_peptides.txt: Normal uniquepeptides")
    print(" - Tumor_unique_peptides.txt: Tumor uniquepeptides")
    print("=" * 80)
    print("analyzecompleted!")
    print("=" * 80)
    return df_results, global_stats

if __name__ == "__main__":
    df_results, global_stats = main()

## R NR

In [ ]:
import pandas as pd
import gzip
from pathlib import Path

def read_peptides_from_file(file_path):
    """readpeptidedata files, returnpeptidetext"""
    peptides = set()
    try:
        if file_path.endswith('.gz'):
            with gzip.open(file_path, 'rt') as f:
                df = pd.read_csv(f, sep='\t')
        else:
            df = pd.read_csv(file_path, sep='\t')
            if 'peptide' in df.columns:
                peptides = set(df['peptide'].dropna().unique())
    except Exception as e:
        print(f" Error reading {file_path}: {e}")
        return peptides

def get_sample_peptides(sample_name, hlai_dir, hlaii_dir):
    """textsamplesallpeptides(HLAIandHLAIItext)"""
    all_peptides = set()
    # readHLAIdata
    hlai_file = Path(hlai_dir) / f"{sample_name}.pHLA_I.tsv.gz"
    if hlai_file.exists():
        peptides = read_peptides_from_file(str(hlai_file))
        all_peptides.update(peptides)
        print(f" {hlai_file.name}: {len(peptides)} peptides")
    else:
        print(f" Warning: {hlai_file.name} textexists")
        # readHLAIIdata
        hlaii_file = Path(hlaii_dir) / f"{sample_name}.pHLA_II.tsv.gz"
        if hlaii_file.exists():
            peptides = read_peptides_from_file(str(hlaii_file))
            all_peptides.update(peptides)
            print(f" {hlaii_file.name}: {len(peptides)} peptides")
        else:
            print(f" Warning: {hlaii_file.name} textexists")
            return all_peptides

def get_group_peptides(sample_list, hlai_dir, hlaii_dir, group_name):
    """text(RorNR)allsamplepeptide"""
    group_peptides = set()
    print(f"\nprocesstext: {group_name}")
    print(f"sample count: {len(sample_list)}")
    for sample_name in sample_list:
        print(f"\nsample: {sample_name}")
        sample_peptides = get_sample_peptides(sample_name, hlai_dir, hlaii_dir)
        print(f" sample {sample_name} text: {len(sample_peptides)} uniquepeptide")
        group_peptides.update(sample_peptides)
        return group_peptides

def main():
    # Path configuration
    hlai_dir = "/path/to/project/data_V3/peptides/pathseq_reads_peptides_inhouse/filtered_out/BindLevel/HLAI"
    hlaii_dir = "/path/to/project/data_V3/peptides/pathseq_reads_peptides_inhouse/filtered_out/BindLevel/HLAII"
    # textsampletext
    sample_groups = {
        'NR': ['B2', 'B3', 'B6', 'C6'],
        'R': ['B1', 'B4', 'B7', 'E5']
    }
    # textRtextandNRtextpeptide
    print("="*80)
    r_peptides = get_group_peptides(sample_groups['R'], hlai_dir, hlaii_dir, 'R')
    print(f"\nRtext: {len(r_peptides)} uniquepeptide")
    print("\n" + "="*80)
    nr_peptides = get_group_peptides(sample_groups['NR'], hlai_dir, hlaii_dir, 'NR')
    print(f"\nNRtext: {len(nr_peptides)} uniquepeptide")
    # calculatetextdata
    print("\n" + "="*80)
    print("textstatistics:")
    print("="*80)
    # textpeptide
    shared_peptides = r_peptides & nr_peptides
    # R-onlypeptide
    r_only_peptides = r_peptides - nr_peptides
    # NR-onlypeptide
    nr_only_peptides = nr_peptides - r_peptides
    # text
    total_peptides = len(r_only_peptides) + len(shared_peptides) + len(nr_only_peptides)
    # calculatetext
    if total_peptides > 0:
        r_only_pct = (len(r_only_peptides) / total_peptides) * 100
        nr_only_pct = (len(nr_only_peptides) / total_peptides) * 100
        shared_pct = (len(shared_peptides) / total_peptides) * 100
    else:
        r_only_pct = nr_only_pct = shared_pct = 0.0
        # outputresults
        print(f"\nR-onlypeptide: {len(r_only_peptides):,} ({r_only_pct:.2f}%)")
        print(f"NR-onlypeptide: {len(nr_only_peptides):,} ({nr_only_pct:.2f}%)")
        print(f"textpeptide: {len(shared_peptides):,} ({shared_pct:.2f}%)")
        print(f"text: {total_peptides:,}")
        # textand
        total_pct = r_only_pct + nr_only_pct + shared_pct
        print(f"textand: {total_pct:.2f}% (text100.00%)")
        # Create output directory
        output_dir = Path("/path/to/project/results_V3/summary/shareing_rate_peptide/R_NR_comparison")
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"\noutput directory: {output_dir}")
        # saveresults
        results = {
            'Category': ['R_only', 'NR_only', 'Shared', 'Total'],
            'Count': [len(r_only_peptides), len(nr_only_peptides), len(shared_peptides), total_peptides],
            'Percentage': [f"{r_only_pct:.2f}%", f"{nr_only_pct:.2f}%", f"{shared_pct:.2f}%", "100.00%"]
        }
        df_results = pd.DataFrame(results)
        output_file = output_dir / "peptide_venn_statistics.csv"
        df_results.to_csv(output_file, index=False)
        print(f"\nresultssavedto: {output_file}")
        # savedetailedpeptide columntext(optional)
        print("\nsavedetailedpeptide columntext...")
        # R-only
        with open(output_dir / "R_only_peptides.txt", 'w') as f:
            f.write('\n'.join(sorted(r_only_peptides)))
            # NR-only
            with open(output_dir / "NR_only_peptides.txt", 'w') as f:
                f.write('\n'.join(sorted(nr_only_peptides)))
                # text
                with open(output_dir / "Shared_peptides.txt", 'w') as f:
                    f.write('\n'.join(sorted(shared_peptides)))
                    print("detailedpeptide columntextsaved!")
                    # textstatistics
                    print("\n" + "="*80)
                    print("textstatistics:")
                    print(f"Rtextsample: {', '.join(sample_groups['R'])}")
                    print(f"NRtextsample: {', '.join(sample_groups['NR'])}")
                    print(f"Rtexttotal peptidestext: {len(r_peptides):,}")
                    print(f"NRtexttotal peptidestext: {len(nr_peptides):,}")
                    return df_results

if __name__ == "__main__":
    results = main()
    print("\nanalyzecompleted!")

In [ ]:
import pandas as pd
from pathlib import Path

def read_peptides_from_patient(patient_dir):
    """
    readtextclassIandclassIIdata, textpeptide
    onlyreadBindLevelcolumntextrecord
    """
    peptides = set()
    # findtextallfile(classIandclassII)
    patient_files = list(Path(patient_dir).glob("*.netmhc_class*.tsv"))
    for file_path in patient_files:
        try:
            # readTSVfile
            df = pd.read_csv(file_path, sep='\t')
            # check required columnstextexists
            if 'BindLevel' in df.columns and 'Peptide' in df.columns:
                # onlytextBindLeveltextrecord
                df_filtered = df[df['BindLevel'].notna()]
                # extractPeptidecolumntextdeduplicate
                patient_peptides = set(df_filtered['Peptide'].dropna().unique())
                peptides.update(patient_peptides)
                print(f" read {file_path.name}: {len(patient_peptides)} peptides")
        except Exception as e:
            print(f" Error reading {file_path}: {e}")
            return peptides

def get_group_peptides(group_dir):
    """
    text(RorNR)alltextpeptide
    """
    group_peptides = set()
    group_path = Path(group_dir)
    # textalltextdirectory/file
    # texteachtextfiletextdirectorytext
    patient_files = {}
    for file_path in group_path.glob("*.netmhc_class*.tsv"):
        # fromfilenameextracttextID(textformatfor PatientID.netmhc_classX.tsv)
        patient_id = file_path.name.split('.')[0]
        if patient_id not in patient_files:
            patient_files[patient_id] = []
            patient_files[patient_id].append(file_path)
            print(f"\nprocesstext: {group_path.name}")
            print(f"found {len(patient_files)} text")
            # processeachtext
            for patient_id, files in patient_files.items():
                print(f"\ntext: {patient_id}")
                patient_peptides = set()
                for file_path in files:
                    try:
                        df = pd.read_csv(file_path, sep='\t')
                        if 'BindLevel' in df.columns and 'Peptide' in df.columns:
                            df_filtered = df[df['BindLevel'].notna()]
                            file_peptides = set(df_filtered['Peptide'].dropna().unique())
                            patient_peptides.update(file_peptides)
                            print(f" {file_path.name}: {len(file_peptides)} peptides")
                    except Exception as e:
                        print(f" Error reading {file_path}: {e}")
                        print(f" text {patient_id} text: {len(patient_peptides)} uniquepeptide")
                        group_peptides.update(patient_peptides)
                        return group_peptides

def main():
    # Path configuration
    r_group_dir = "/path/to/project/results/special/CRC_paired_V2.4/06.antigen/06.1.data/R"
    nr_group_dir = "/path/to/project/results/special/CRC_paired_V2.4/06.antigen/06.1.data/NR"
    # textRtextandNRtextpeptide
    print("="*80)
    r_peptides = get_group_peptides(r_group_dir)
    print(f"\nRtext: {len(r_peptides)} uniquepeptide")
    print("\n" + "="*80)
    nr_peptides = get_group_peptides(nr_group_dir)
    print(f"\nNRtext: {len(nr_peptides)} uniquepeptide")
    # calculatetextdata
    print("\n" + "="*80)
    print("textstatistics:")
    print("="*80)
    # textpeptide
    shared_peptides = r_peptides & nr_peptides
    # R-onlypeptide
    r_only_peptides = r_peptides - nr_peptides
    # NR-onlypeptide
    nr_only_peptides = nr_peptides - r_peptides
    # text
    total_peptides = len(r_only_peptides) + len(shared_peptides) + len(nr_only_peptides)
    # calculatetext
    if total_peptides > 0:
        r_only_pct = (len(r_only_peptides) / total_peptides) * 100
        nr_only_pct = (len(nr_only_peptides) / total_peptides) * 100
        shared_pct = (len(shared_peptides) / total_peptides) * 100
    else:
        r_only_pct = nr_only_pct = shared_pct = 0.0
        # outputresults
        print(f"\nR-onlypeptide: {len(r_only_peptides):,} ({r_only_pct:.2f}%)")
        print(f"NR-onlypeptide: {len(nr_only_peptides):,} ({nr_only_pct:.2f}%)")
        print(f"textpeptide: {len(shared_peptides):,} ({shared_pct:.2f}%)")
        print(f"text: {total_peptides:,}")
        # Create output directory
        output_dir = Path("/path/to/project/results/special/CRC_V2.8/06.antigen/06.1.data")
        output_dir.mkdir(parents=True, exist_ok=True)
        print(f"\noutput directory: {output_dir}")
        # saveresults
        results = {
            'Category': ['R_only', 'NR_only', 'Shared', 'Total'],
            'Count': [len(r_only_peptides), len(nr_only_peptides), len(shared_peptides), total_peptides],
            'Percentage': [f"{r_only_pct:.2f}%", f"{nr_only_pct:.2f}%", f"{shared_pct:.2f}%", "100.00%"]
        }
        df_results = pd.DataFrame(results)
        output_file = output_dir / "peptide_venn_statistics.csv"
        df_results.to_csv(output_file, index=False)
        print(f"\nresultssavedto: {output_file}")
        # savedetailedpeptide columntext(optional)
        print("\nsavedetailedpeptide columntext...")
        # R-only
        with open(output_dir / "R_only_peptides.txt", 'w') as f:
            f.write('\n'.join(sorted(r_only_peptides)))
            # NR-only
            with open(output_dir / "NR_only_peptides.txt", 'w') as f:
                f.write('\n'.join(sorted(nr_only_peptides)))
                # text
                with open(output_dir / "Shared_peptides.txt", 'w') as f:
                    f.write('\n'.join(sorted(shared_peptides)))
                    print("detailedpeptide columntextsaved!")
                    return df_results

if __name__ == "__main__":
    results = main()
    print("\nanalyzecompleted!")

## sharing rate in common species

### textshare species

In [ ]:
import pandas as pd
import os

# allcancer typetext
cancer_types = ["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA"]

# filepathtext - textfor .csv.gz format
base_path = "/path/to/project/results_V3/cancers_V3.1/{}/01.group_matrix/{}_{}_species_pathseq.csv"

# output directory
output_dir = "/path/to/project/results_V3/summary/sharing_rate"
os.makedirs(output_dir, exist_ok=True)

def get_microbes_50pct_nonzero(file_path):
    """readcsv.gzfiletextreturntext50%textrowstextmicrobetext"""
    # pandas textread .gz textfile
    df = pd.read_csv(file_path)
    sample_data = df.iloc[:, 1:]
    threshold = sample_data.shape[1] * 0.5
    df_filtered = df[sample_data.ne(0).sum(axis=1) >= threshold]
    microbes = df_filtered.iloc[:, 0]
    return set(microbes)

# ========== text: textsharedmicrobe ==========
print("===== calculatetextsharedmicrobe =====")

# textallcancer type normal and tumor microbe
all_normal_microbes = set()
all_tumor_microbes = set()

for cancer in cancer_types:
    try:
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")

        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)

        all_normal_microbes.update(normal_microbes)
        all_tumor_microbes.update(tumor_microbes)
        print(f" {cancer}: Normal={len(normal_microbes)}, Tumor={len(tumor_microbes)}")
    except Exception as e:
        print(f" process {cancer} text: {e}")

# calculatetext
pan_cancer_intersection = all_normal_microbes & all_tumor_microbes

print(f"\ntext:")
print(f" Total Normal microbetext: {len(all_normal_microbes)}")
print(f" Total Tumor microbetext: {len(all_tumor_microbes)}")
print(f" shared microbe count: {len(pan_cancer_intersection)}")

# savetextsharedmicrobe list
pan_cancer_file = os.path.join(output_dir, "PanCancer_shared_microbes.txt")
with open(pan_cancer_file, 'w') as f:
    for microbe in sorted(pan_cancer_intersection):
        f.write(f"{microbe}\n")
print(f"\nOK savedtextsharedmicrobe listto: {pan_cancer_file}")


# ========== text: textcancer typetextsharedmicrobe ==========
print("\n===== calculatetextcancer typetextsharedmicrobe =====")

for cancer in cancer_types:
    try:
        # buildpath
        normal_file = base_path.format(cancer, cancer, "Normal")
        tumor_file = base_path.format(cancer, cancer, "Tumor")

        # readdata
        normal_microbes = get_microbes_50pct_nonzero(normal_file)
        tumor_microbes = get_microbes_50pct_nonzero(tumor_file)

        # calculatetextcancer typetext
        cancer_intersection = normal_microbes & tumor_microbes

        print(f" {cancer}: Normal={len(normal_microbes)}, Tumor={len(tumor_microbes)}, shared={len(cancer_intersection)}")

        # savetextcancer typesharedmicrobe list
        cancer_file = os.path.join(output_dir, f"{cancer}_shared_microbes.txt")
        with open(cancer_file, 'w') as f:
            for microbe in sorted(cancer_intersection):
                f.write(f"{microbe}\n")
                print(f" OK savedto: {cancer_file}")

    except Exception as e:
        print(f" process {cancer} text: {e}")

print(f"\n===== completed! allfilesavedto {output_dir} =====")
print(f"generated total {len(cancer_types) + 1} txtfile")

### textaftertextsharetext

In [ ]:
# import os
# import gzip
# import pandas as pd
# from collections import defaultdict
# from typing import Dict, List, Set, Tuple
# import re

# class PeptideSharingAnalyzer:
# def __init__(self, base_dir: str):
# self.base_dir = base_dir
# self.microbe_to_taxids = {}
# self.cancer_microbes = {}
# self.cancer_samples = {}
# self.sample_cache = {} # textsample counttext, textread
# def extract_taxids_from_pairs(self, pairs_str: str) -> Set[str]:
# """
# frompairstextinextractalltaxid
# text: pairs_strtextrecord, text';'text
# eachrecordtexttaxid, text','text
# """
# taxids = set()
# # textpairs_strtext, textreturntext
# if not isinstance(pairs_str, str):
# return taxids
# # textrecord
# records = pairs_str.split(';')
# for record in records:
# record = record.strip()
# if not record:
# continue
# # texteachrecordinfindCTAXAtext
# if 'CTAXA:' in record:
# # textextract
# # findCTAXA:aftertextandtext
# pattern = r'CTAXA:([\d,]+)'
# matches = re.findall(pattern, record)
# for match in matches:
# # texttaxid
# for taxid in match.split(','):
# taxid = taxid.strip()
# if taxid and taxid.isdigit(): # text
# taxids.add(taxid)
# return taxids
# def load_all_data(self):
# """loadallrequireddata"""
# # loadtaxidmapping
# taxid_file = os.path.join(self.base_dir, "data/genome_annotation/V7_taxid_hierarchy.txt")
# self._load_taxid_mapping(taxid_file)
# # loadtextmicrobe list
# microbes_dir = os.path.join(self.base_dir, "results_V3/summary/sharing_rate")
# self._load_shared_microbes(microbes_dir)
# # loadsampleinformation
# sample_info_file = os.path.join(self.base_dir, "data_V3/meta/cancer_status_sample_1897.csv")
# self._load_sample_info(sample_info_file)
# def _load_taxid_mapping(self, taxid_file: str):
# """loadtaxidmapping"""
# print(f"loadtaxidmappingfile: {taxid_file}")
# df = pd.read_csv(taxid_file, sep='\t')
# for _, row in df.iterrows():
# species = str(row['species_scientific_name'])
# # textandtextfortext
# processed_name = species.replace(' ', '_').replace('/', '_')
# taxid = str(row['tax_id'])
# if processed_name not in self.microbe_to_taxids:
# self.microbe_to_taxids[processed_name] = []
# if taxid not in self.microbe_to_taxids[processed_name]:
# self.microbe_to_taxids[processed_name].append(taxid)
# print(f"loadtext {len(self.microbe_to_taxids)} microbestaxidmapping")
# def _load_shared_microbes(self, microbes_dir: str):
# """loadtextmicrobe list"""
# print(f"loadtextmicrobe list: {microbes_dir}")
# self.cancer_microbes = {}
# for filename in os.listdir(microbes_dir):
# if filename.endswith('_shared_microbes.txt'):
# cancer = filename.split('_shared_microbes.txt')[0]
# filepath = os.path.join(microbes_dir, filename)
# with open(filepath, 'r') as f:
# microbes = [line.strip() for line in f if line.strip()]
# self.cancer_microbes[cancer] = microbes
# print(f" {cancer}: {len(microbes)} textmicrobe")
# def _load_sample_info(self, sample_info_file: str):
# """loadsampleinformation"""
# print(f"loadsampleinformation: {sample_info_file}")
# df = pd.read_csv(sample_info_file)
# for _, row in df.iterrows():
# cancer = row['Cancer_Type']
# status = row['Status']
# sample = row['Sample']
# if cancer not in self.cancer_samples:
# self.cancer_samples[cancer] = {'Normal': [], 'Tumor': []}
# self.cancer_samples[cancer][status].append(sample)
# # statisticssample counttext
# total_samples = 0
# for cancer, status_dict in self.cancer_samples.items():
# normal_count = len(status_dict.get('Normal', []))
# tumor_count = len(status_dict.get('Tumor', []))
# total_samples += normal_count + tumor_count
# print(f" {cancer}: Normal={normal_count}, Tumor={tumor_count}")
# print(f"text {total_samples} samples")
# def get_microbes_taxids_for_cancer(self, cancer: str) -> Set[str]:
# """textcancer typetextmicrobetaxidtext"""
# if cancer not in self.cancer_microbes:
# return set()
# microbes_taxids = set()
# not_found_count = 0
# for microbe in self.cancer_microbes[cancer]:
# if microbe in self.microbe_to_taxids:
# microbes_taxids.update(self.microbe_to_taxids[microbe])
# else:
# # textformat
# # 1. text
# original_name = microbe.replace('_', ' ')
# if original_name in self.microbe_to_taxids:
# microbes_taxids.update(self.microbe_to_taxids[original_name])
# else:
# not_found_count += 1
# if not_found_count > 0:
# print(f" {cancer}: {not_found_count} microbesnot foundtaxidmapping")
# return microbes_taxids
# def process_sample_peptides(self, sample: str, microbes_taxids: Set[str]) -> Set[str]:
# """
# processtextsamples, textmicrobepeptide
# return:
# Set[str]: peptide sequencetext
# """
# peptides = set()
# # checktext
# cache_key = f"{sample}_{sorted(microbes_taxids)}"
# if cache_key in self.sample_cache:
# return self.sample_cache[cache_key]
# # processHLAIandHLAII
# for hla_type in ['HLAI', 'HLAII']:
# try:
# # 1. readpeptidetextfile
# peptide_file = self._get_peptide_file(sample, hla_type)
# if not os.path.exists(peptide_file):
# continue
# # 2. readsource_idtosseqmapping
# id2seq_dict = self._load_id2seq_mapping(sample)
# if not id2seq_dict:
# continue
# # 3. readsseqtotaxidmapping
# sseq_to_taxids = self._load_sseq2taxid_mapping(sample)
# if not sseq_to_taxids:
# continue
# # 4. processpeptidefile
# # textchunksizetextprocesstext
# chunksize = 50000
# for chunk in pd.read_csv(peptide_file, sep='\t',
# usecols=['peptide', 'source_ids'],
# compression='gzip',
# chunksize=chunksize):
# for _, row in chunk.iterrows():
# peptide = str(row['peptide']).strip()
# if not peptide:
# continue
# source_ids = str(row['source_ids']).split(',')
# # checkeachsource_id
# for source_id in source_ids:
# source_id = source_id.strip()
# if not source_id:
# continue
# # textsseq
# if source_id in id2seq_dict:
# sseq = id2seq_dict[source_id]
# # texttaxid
# if sseq in sseq_to_taxids:
# taxids = sseq_to_taxids[sseq]
# # checktextwithtext
# if taxids.intersection(microbes_taxids):
# peptides.add(peptide)
# break # foundtextmatchedsource_idtext
# except Exception as e:
# print(f" Process sample {sample} {hla_type} datatext: {str(e)}")
# # textresults
# self.sample_cache[cache_key] = peptides
# return peptides
# def _get_peptide_file(self, sample: str, hla_type: str) -> str:
# """textpeptidefilepath"""
# bindlevel_dir = os.path.join(self.base_dir, "data_V3/peptides/pathseq_reads_peptides_public/filtered_out/BindLevel")
# if hla_type == 'HLAI':
# return os.path.join(bindlevel_dir, "HLAI", f"{sample}.pHLA_I.tsv.gz")
# else:
# return os.path.join(bindlevel_dir, "HLAII", f"{sample}.pHLA_II.tsv.gz")

# def _load_id2seq_mapping(self, sample: str) -> Dict[str, str]:
# """loadsource_idtosseqmapping"""
# phla_dir = os.path.join(self.base_dir, "data_V3/peptides/pathseq_reads_peptides_public/pHLA_bare")
# id2seq_file = os.path.join(phla_dir, sample, f"{sample}.id2seq.tsv.gz")
# if not os.path.exists(id2seq_file):
# return {}
# try:
# df = pd.read_csv(id2seq_file, sep='\t', usecols=['Identity', 'sseq'], compression='gzip')
# return dict(zip(df['Identity'], df['sseq']))
# except Exception as e:
# print(f" readid2seqfile {id2seq_file} text: {str(e)}")
# return {}
# def _load_sseq2taxid_mapping(self, sample: str) -> Dict[str, Set[str]]:
# """loadsseqtotaxidmapping"""
# phla_dir = os.path.join(self.base_dir, "data_V3/peptides/pathseq_reads_peptides_public/pHLA_bare")
# sseq2pairs_file = os.path.join(phla_dir, sample, f"{sample}.sseq2pairs.tsv.gz")
# if not os.path.exists(sseq2pairs_file):
# return {}
# sseq_to_taxids = {}
# try:
# # textgziptextfile
# with gzip.open(sseq2pairs_file, 'rt') as f:
# for line_num, line in enumerate(f, 1):
# line = line.strip()
# if not line:
# continue
# parts = line.split('\t')
# if len(parts) < 2:
# continue
# sseq = parts[0].strip()
# pairs_str = parts[1]
# # extracttaxid
# taxids = self.extract_taxids_from_pairs(pairs_str)
# if taxids:
# sseq_to_taxids[sseq] = taxids
# # text(text10000rows)
# if line_num % 10000 == 0:
# print(f" processed {line_num} rows...")
# except Exception as e:
# print(f" readsseq2pairsfile {sseq2pairs_file} text: {str(e)}")
# return sseq_to_taxids
# def analyze_cancer(self, cancer: str, max_samples_per_status: int = 10) -> Dict:
# """analyzetextcancer typepeptidetext"""
# print(f"\nanalyzecancer type: {cancer}")
# # 1. textmicrobetaxid
# microbes_taxids = self.get_microbes_taxids_for_cancer(cancer)
# if not microbes_taxids:
# print(f" error: not foundwithtexttaxid")
# return None
# print(f" texttaxidcount: {len(microbes_taxids)}")
# # 2. textsample columntext
# if cancer == 'PanCancer':
# # text: textallsample
# normal_samples = []
# tumor_samples = []
# for c_type, status_dict in self.cancer_samples.items():
# if c_type != 'PanCancer': # text
# normal_samples.extend(status_dict.get('Normal', []))
# tumor_samples.extend(status_dict.get('Tumor', []))
# else:
# # textcancer type
# if cancer not in self.cancer_samples:
# print(f" error: cancer type {cancer} textsampleinformationinnot found")
# return None
# normal_samples = self.cancer_samples[cancer].get('Normal', [])[:max_samples_per_status]
# tumor_samples = self.cancer_samples[cancer].get('Tumor', [])[:max_samples_per_status]
# print(f" Normalsample: {len(normal_samples)} ")
# print(f" Tumorsample: {len(tumor_samples)} ")
# # 3. textpeptidetext
# print(" processNormalsamplepeptide...")
# normal_peptides = set()
# for i, sample in enumerate(normal_samples, 1):
# print(f" processNormalsample {i}/{len(normal_samples)}: {sample}")
# peptides = self.process_sample_peptides(sample, microbes_taxids)
# normal_peptides.update(peptides)
# print(f" textfirstNormalpeptidetext: {len(normal_peptides)}")
# print(" processTumorsamplepeptide...")
# tumor_peptides = set()
# for i, sample in enumerate(tumor_samples, 1):
# print(f" processTumorsample {i}/{len(tumor_samples)}: {sample}")
# peptides = self.process_sample_peptides(sample, microbes_taxids)
# tumor_peptides.update(peptides)
# print(f" textfirstTumorpeptidetext: {len(tumor_peptides)}")
# # 4. calculatetext
# shared_peptides = normal_peptides.intersection(tumor_peptides)
# normal_unique = normal_peptides - tumor_peptides
# tumor_unique = tumor_peptides - normal_peptides
# # 5. calculatetext
# total = len(shared_peptides) + len(normal_unique) + len(tumor_unique)
# if total > 0:
# shared_percent = (len(shared_peptides) / total * 100)
# normal_percent = (len(normal_unique) / total * 100)
# tumor_percent = (len(tumor_unique) / total * 100)
# else:
# shared_percent = normal_percent = tumor_percent = 0.0
# # formattext
# def format_percent(value):
# return f"{value:.1f}%"
# result = {
# 'Cancer': cancer,
# 'Normal_Samples': len(normal_samples),
# 'Tumor_Samples': len(tumor_samples),
# 'Shared_Microbes': len(self.cancer_microbes.get(cancer, [])),
# 'Total_Peptides': total,
# 'Shared_Peptides': len(shared_peptides),
# 'Shared_Percent': format_percent(shared_percent),
# 'Normal_Unique': len(normal_unique),
# 'Normal_Unique_Percent': format_percent(normal_percent),
# 'Tumor_Unique': len(tumor_unique),
# 'Tumor_Unique_Percent': format_percent(tumor_percent),
# 'Normal_Total': len(normal_peptides),
# 'Tumor_Total': len(tumor_peptides)
# }
# print(f" results:")
# print(f" Normaltotal peptides: {len(normal_peptides)}")
# print(f" Tumortotal peptides: {len(tumor_peptides)}")
# print(f" shared peptides: {len(shared_peptides)} ({result['Shared_Percent']})")
# print(f" Normal unique: {len(normal_unique)} ({result['Normal_Unique_Percent']})")
# print(f" Tumor unique: {len(tumor_unique)} ({result['Tumor_Unique_Percent']})")
# return result
# def run_analysis(self, cancers_to_analyze=None, max_samples=10):
# """textrowstextanalyze"""
# print("=" * 80)
# print("startpeptidetextanalyze")
# print("=" * 80)
# # loadalldata
# self.load_all_data()
# # textanalyzecancer type
# if cancers_to_analyze is None:
# cancers_to_analyze = list(self.cancer_microbes.keys())
# results = []
# pan_cancer_stats = {
# 'Normal': set(),
# 'Tumor': set(),
# 'Shared': set()
# }
# # analyzeeachcancer type
# for cancer in cancers_to_analyze:
# result = self.analyze_cancer(cancer, max_samples_per_status=max_samples)
# if result:
# results.append(result)
# # Updatetextstatistics
# if cancer != 'PanCancer':
# # text: textprocess, actualtextcalculate
# pass
# # createresultsDataFrame
# df = pd.DataFrame(results)
# # saveresults
# output_dir = os.path.join(self.base_dir, "results/peptide_analysis")
# os.makedirs(output_dir, exist_ok=True)
# output_file = os.path.join(output_dir, "peptide_sharing_results.csv")
# df.to_csv(output_file, index=False)
# print(f"\nresultssavedto: {output_file}")
# # savedetailedstatistics
# summary_file = os.path.join(output_dir, "peptide_sharing_summary.txt")
# with open(summary_file, 'w') as f:
# f.write("Peptide-sharing analysis summary\n")
# f.write("=" * 80 + "\n\n")
# for _, row in df.iterrows():
# f.write(f"cancer type: {row['Cancer']}\n")
# f.write(f" sample count: Normal={row['Normal_Samples']}, Tumor={row['Tumor_Samples']}\n")
# f.write(f" shared microbe count: {row['Shared_Microbes']}\n")
# f.write(f" total peptide types: {row['Total_Peptides']}\n")
# f.write(f" shared peptides: {row['Shared_Peptides']} ({row['Shared_Percent']})\n")
# f.write(f" Normal unique: {row['Normal_Unique']} ({row['Normal_Unique_Percent']})\n")
# f.write(f" Tumor unique: {row['Tumor_Unique']} ({row['Tumor_Unique_Percent']})\n")
# f.write("-" * 60 + "\n")
# print(f"detailedstatisticssavedto: {summary_file}")
# return df

# def main():
# """text"""
# base_dir = "/path/to/project"
# analyzer = PeptideSharingAnalyzer(base_dir)
# # textrowsanalyze(textsample)
# results_df = analyzer.run_analysis(
# cancers_to_analyze=["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG", "KIDNEY", "BLCA", "STAD", "LIHC", "THCA", 'PanCancer'], # textcancer type
# max_samples=None #)
# # textresults
# print("\n" + "=" * 80)
# print("textresultstext:")
# print("=" * 80)
# print(results_df.to_string())

# if __name__ == "__main__":
# main()

In [ ]:
import os
import gzip
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Set, Tuple
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
import multiprocessing as mp

class PeptideSharingAnalyzer:
    def __init__(self, base_dir: str, n_workers: int = None):
        self.base_dir = base_dir
        self.microbe_to_taxids = {}
        self.cancer_microbes = {}
        self.cancer_samples = {}
        # textrowsprocesstext(text)
        # self.sample_cache = {}
        # text, textalltext
        if n_workers is None:
            self.n_workers = mp.cpu_count()
        else:
            self.n_workers = min(n_workers, mp.cpu_count())
            print(f"text {self.n_workers} textrowstextrowsprocess")
            def extract_taxids_from_pairs(self, pairs_str: str) -> Set[str]:
                """frompairstextinextractalltaxid"""
                taxids = set()
                if not isinstance(pairs_str, str):
                    return taxids
                records = pairs_str.split(';')
                for record in records:
                    record = record.strip()
                    if not record:
                        continue
                    if 'CTAXA:' in record:
                        pattern = r'CTAXA:([\d,]+)'
                        matches = re.findall(pattern, record)
                        for match in matches:
                            for taxid in match.split(','):
                                taxid = taxid.strip()
                                if taxid and taxid.isdigit():
                                    taxids.add(taxid)
                                    return taxids
                                def load_all_data(self):
                                    """loadallrequireddata"""
                                    taxid_file = os.path.join(self.base_dir, "data/genome_annotation/V7_taxid_hierarchy.txt")
                                    self._load_taxid_mapping(taxid_file)
                                    microbes_dir = os.path.join(self.base_dir, "results_V3/summary/sharing_rate")
                                    self._load_shared_microbes(microbes_dir)
                                    sample_info_file = os.path.join(self.base_dir, "data_V3/meta/cancer_status_sample_1897.csv")
                                    self._load_sample_info(sample_info_file)
                                    def _load_taxid_mapping(self, taxid_file: str):
                                        """loadtaxidmapping"""
                                        print(f"loadtaxidmappingfile: {taxid_file}")
                                        df = pd.read_csv(taxid_file, sep='\t')
                                        for _, row in df.iterrows():
                                            species = str(row['species_scientific_name'])
                                            processed_name = species.replace(' ', '_').replace('/', '_')
                                            taxid = str(row['tax_id'])
                                            if processed_name not in self.microbe_to_taxids:
                                                self.microbe_to_taxids[processed_name] = []
                                                if taxid not in self.microbe_to_taxids[processed_name]:
                                                    self.microbe_to_taxids[processed_name].append(taxid)
                                                    print(f"loadtext {len(self.microbe_to_taxids)} microbestaxidmapping")
                                                    def _load_shared_microbes(self, microbes_dir: str):
                                                        """loadtextmicrobe list"""
                                                        print(f"loadtextmicrobe list: {microbes_dir}")
                                                        self.cancer_microbes = {}
                                                        for filename in os.listdir(microbes_dir):
                                                            if filename.endswith('_shared_microbes.txt'):
                                                                cancer = filename.split('_shared_microbes.txt')[0]
                                                                filepath = os.path.join(microbes_dir, filename)
                                                                with open(filepath, 'r') as f:
                                                                    microbes = [line.strip() for line in f if line.strip()]
                                                                    self.cancer_microbes[cancer] = microbes
                                                                    print(f" {cancer}: {len(microbes)} textmicrobe")
                                                                    def _load_sample_info(self, sample_info_file: str):
                                                                        """loadsampleinformation"""
                                                                        print(f"loadsampleinformation: {sample_info_file}")
                                                                        df = pd.read_csv(sample_info_file)
                                                                        for _, row in df.iterrows():
                                                                            cancer = row['Cancer_Type']
                                                                            status = row['Status']
                                                                            sample = row['Sample']
                                                                            if cancer not in self.cancer_samples:
                                                                                self.cancer_samples[cancer] = {'Normal': [], 'Tumor': []}
                                                                                self.cancer_samples[cancer][status].append(sample)
                                                                                total_samples = 0
                                                                                for cancer, status_dict in self.cancer_samples.items():
                                                                                    normal_count = len(status_dict.get('Normal', []))
                                                                                    tumor_count = len(status_dict.get('Tumor', []))
                                                                                    total_samples += normal_count + tumor_count
                                                                                    print(f" {cancer}: Normal={normal_count}, Tumor={tumor_count}")
                                                                                    print(f"text {total_samples} samples")
                                                                                    def get_microbes_taxids_for_cancer(self, cancer: str) -> Set[str]:
                                                                                        """textcancer typetextmicrobetaxidtext"""
                                                                                        if cancer not in self.cancer_microbes:
                                                                                            return set()
                                                                                        microbes_taxids = set()
                                                                                        not_found_count = 0
                                                                                        for microbe in self.cancer_microbes[cancer]:
                                                                                            if microbe in self.microbe_to_taxids:
                                                                                                microbes_taxids.update(self.microbe_to_taxids[microbe])
                                                                                            else:
                                                                                                original_name = microbe.replace('_', ' ')
                                                                                                if original_name in self.microbe_to_taxids:
                                                                                                    microbes_taxids.update(self.microbe_to_taxids[original_name])
                                                                                                else:
                                                                                                    not_found_count += 1
                                                                                                    if not_found_count > 0:
                                                                                                        print(f" {cancer}: {not_found_count} microbesnot foundtaxidmapping")
                                                                                                        return microbes_taxids
                                                                                                    def analyze_cancer(self, cancer: str, max_samples_per_status: int = None) -> Dict:
                                                                                                        """analyzetextcancer typepeptidetext(textrowstext)"""
                                                                                                        print(f"\nanalyzecancer type: {cancer}")
                                                                                                        # 1. textmicrobetaxid
                                                                                                        microbes_taxids = self.get_microbes_taxids_for_cancer(cancer)
                                                                                                        if not microbes_taxids:
                                                                                                            print(f" error: not foundwithtexttaxid")
                                                                                                            return None
                                                                                                        print(f" texttaxidcount: {len(microbes_taxids)}")
                                                                                                        # 2. textsample columntext
                                                                                                        if cancer == 'PanCancer':
                                                                                                            normal_samples = []
                                                                                                            tumor_samples = []
                                                                                                            for c_type, status_dict in self.cancer_samples.items():
                                                                                                                if c_type != 'PanCancer':
                                                                                                                    normal_samples.extend(status_dict.get('Normal', []))
                                                                                                                    tumor_samples.extend(status_dict.get('Tumor', []))
                                                                                                                else:
                                                                                                                    if cancer not in self.cancer_samples:
                                                                                                                        print(f" error: cancer type {cancer} textsampleinformationinnot found")
                                                                                                                        return None
                                                                                                                    if max_samples_per_status:
                                                                                                                        normal_samples = self.cancer_samples[cancer].get('Normal', [])[:max_samples_per_status]
                                                                                                                        tumor_samples = self.cancer_samples[cancer].get('Tumor', [])[:max_samples_per_status]
                                                                                                                    else:
                                                                                                                        normal_samples = self.cancer_samples[cancer].get('Normal', [])
                                                                                                                        tumor_samples = self.cancer_samples[cancer].get('Tumor', [])
                                                                                                                        print(f" Normalsample: {len(normal_samples)} ")
                                                                                                                        print(f" Tumorsample: {len(tumor_samples)} ")
                                                                                                                        # 3. textrowsProcess sample
                                                                                                                        print(" textrowsprocessNormalsamplepeptide...")
                                                                                                                        normal_peptides = self._process_samples_parallel(normal_samples, microbes_taxids, "Normal")
                                                                                                                        print(" textrowsprocessTumorsamplepeptide...")
                                                                                                                        tumor_peptides = self._process_samples_parallel(tumor_samples, microbes_taxids, "Tumor")
                                                                                                                        # 4. calculatetext
                                                                                                                        shared_peptides = normal_peptides.intersection(tumor_peptides)
                                                                                                                        normal_unique = normal_peptides - tumor_peptides
                                                                                                                        tumor_unique = tumor_peptides - normal_peptides
                                                                                                                        # 5. calculatetext
                                                                                                                        total = len(shared_peptides) + len(normal_unique) + len(tumor_unique)
                                                                                                                        if total > 0:
                                                                                                                            shared_percent = (len(shared_peptides) / total * 100)
                                                                                                                            normal_percent = (len(normal_unique) / total * 100)
                                                                                                                            tumor_percent = (len(tumor_unique) / total * 100)
                                                                                                                        else:
                                                                                                                            shared_percent = normal_percent = tumor_percent = 0.0
                                                                                                                            def format_percent(value):
                                                                                                                                return f"{value:.1f}%"
                                                                                                                            result = {
                                                                                                                                'Cancer': cancer,
                                                                                                                                'Normal_Samples': len(normal_samples),
                                                                                                                                'Tumor_Samples': len(tumor_samples),
                                                                                                                                'Shared_Microbes': len(self.cancer_microbes.get(cancer, [])),
                                                                                                                                'Total_Peptides': total,
                                                                                                                                'Shared_Peptides': len(shared_peptides),
                                                                                                                                'Shared_Percent': format_percent(shared_percent),
                                                                                                                                'Normal_Unique': len(normal_unique),
                                                                                                                                'Normal_Unique_Percent': format_percent(normal_percent),
                                                                                                                                'Tumor_Unique': len(tumor_unique),
                                                                                                                                'Tumor_Unique_Percent': format_percent(tumor_percent),
                                                                                                                                'Normal_Total': len(normal_peptides),
                                                                                                                                'Tumor_Total': len(tumor_peptides)
                                                                                                                            }
                                                                                                                            print(f" results:")
                                                                                                                            print(f" Normaltotal peptides: {len(normal_peptides)}")
                                                                                                                            print(f" Tumortotal peptides: {len(tumor_peptides)}")
                                                                                                                            print(f" shared peptides: {len(shared_peptides)} ({result['Shared_Percent']})")
                                                                                                                            print(f" Normal unique: {len(normal_unique)} ({result['Normal_Unique_Percent']})")
                                                                                                                            print(f" Tumor unique: {len(tumor_unique)} ({result['Tumor_Unique_Percent']})")
                                                                                                                            return result
                                                                                                                        def _process_samples_parallel(self, samples: List[str], microbes_taxids: Set[str],                                                                                                                             sample_type: str) -> Set[str]:
                                                                                                                        """textrowsprocesstextsamples"""
                                                                                                                        all_peptides = set()
                                                                                                                        completed = 0
                                                                                                                        total = len(samples)
                                                                                                                        # createtext
                                                                                                                        with ProcessPoolExecutor(max_workers=self.n_workers) as executor:
                                                                                                                            # textalltext
                                                                                                                            future_to_sample = {
                                                                                                                                executor.submit(process_single_sample, sample, self.base_dir, microbes_taxids): sample
                                                                                                                                for sample in samples
                                                                                                                            }
                                                                                                                            # textresults
                                                                                                                            for future in as_completed(future_to_sample):
                                                                                                                                sample = future_to_sample[future]
                                                                                                                                completed += 1
                                                                                                                                try:
                                                                                                                                    peptides = future.result()
                                                                                                                                    all_peptides.update(peptides)
                                                                                                                                    print(f" [{completed}/{total}] completed {sample_type} sample {sample}: "
                                                                                                                                        f"peptide count={len(peptides)}, cumulative={len(all_peptides)}")
                                                                                                                                except Exception as e:
                                                                                                                                    print(f" [{completed}/{total}] process {sample_type} sample {sample} text: {str(e)}")
                                                                                                                                    return all_peptides
                                                                                                                                def run_analysis(self, cancers_to_analyze=None, max_samples=None):
                                                                                                                                    """textrowstextanalyze"""
                                                                                                                                    print("=" * 80)
                                                                                                                                    print("startpeptidetextanalyze(textrowstext)")
                                                                                                                                    print("=" * 80)
                                                                                                                                    self.load_all_data()
                                                                                                                                    if cancers_to_analyze is None:
                                                                                                                                        cancers_to_analyze = list(self.cancer_microbes.keys())
                                                                                                                                        results = []
                                                                                                                                        for cancer in cancers_to_analyze:
                                                                                                                                            result = self.analyze_cancer(cancer, max_samples_per_status=max_samples)
                                                                                                                                            if result:
                                                                                                                                                results.append(result)
                                                                                                                                                df = pd.DataFrame(results)
                                                                                                                                                output_dir = os.path.join(self.base_dir, "results/peptide_analysis")
                                                                                                                                                os.makedirs(output_dir, exist_ok=True)
                                                                                                                                                output_file = os.path.join(output_dir, "peptide_sharing_results.csv")
                                                                                                                                                df.to_csv(output_file, index=False)
                                                                                                                                                print(f"\nresultssavedto: {output_file}")
                                                                                                                                                summary_file = os.path.join(output_dir, "peptide_sharing_summary.txt")
                                                                                                                                                with open(summary_file, 'w') as f:
                                                                                                                                                    f.write("Peptide-sharing analysis summary\n")
                                                                                                                                                    f.write("=" * 80 + "\n\n")
                                                                                                                                                    for _, row in df.iterrows():
                                                                                                                                                        f.write(f"cancer type: {row['Cancer']}\n")
                                                                                                                                                        f.write(f" sample count: Normal={row['Normal_Samples']}, Tumor={row['Tumor_Samples']}\n")
                                                                                                                                                        f.write(f" shared microbe count: {row['Shared_Microbes']}\n")
                                                                                                                                                        f.write(f" total peptide types: {row['Total_Peptides']}\n")
                                                                                                                                                        f.write(f" shared peptides: {row['Shared_Peptides']} ({row['Shared_Percent']})\n")
                                                                                                                                                        f.write(f" Normal unique: {row['Normal_Unique']} ({row['Normal_Unique_Percent']})\n")
                                                                                                                                                        f.write(f" Tumor unique: {row['Tumor_Unique']} ({row['Tumor_Unique_Percent']})\n")
                                                                                                                                                        f.write("-" * 60 + "\n")
                                                                                                                                                        print(f"detailedstatisticssavedto: {summary_file}")
                                                                                                                                                        return df


# text, textrowsprocess
def process_single_sample(sample: str, base_dir: str, microbes_taxids: Set[str]) -> Set[str]:
    """
    processtextsamplestext(text)
    text:
    sample: sampleID
    base_dir: textdirectory
    microbes_taxids: textmicrobetaxidtext
    return:
    Set[str]: peptide sequencetext
    """
    peptides = set()
    # processHLAIandHLAII
    for hla_type in ['HLAI', 'HLAII']:
        try:
            # 1. textpeptidefilepath
            bindlevel_dir = os.path.join(base_dir, "data_V3/peptides/pathseq_reads_peptides_public/filtered_out/BindLevel")
            if hla_type == 'HLAI':
                peptide_file = os.path.join(bindlevel_dir, "HLAI", f"{sample}.pHLA_I.tsv.gz")
            else:
                peptide_file = os.path.join(bindlevel_dir, "HLAII", f"{sample}.pHLA_II.tsv.gz")
                if not os.path.exists(peptide_file):
                    continue
                # 2. readsource_idtosseqmapping
                phla_dir = os.path.join(base_dir, "data_V3/peptides/pathseq_reads_peptides_public/pHLA_bare")
                id2seq_file = os.path.join(phla_dir, sample, f"{sample}.id2seq.tsv.gz")
                if not os.path.exists(id2seq_file):
                    continue
                df_id2seq = pd.read_csv(id2seq_file, sep='\t', usecols=['Identity', 'sseq'], compression='gzip')
                id2seq_dict = dict(zip(df_id2seq['Identity'], df_id2seq['sseq']))
                # 3. readsseqtotaxidmapping
                sseq2pairs_file = os.path.join(phla_dir, sample, f"{sample}.sseq2pairs.tsv.gz")
                if not os.path.exists(sseq2pairs_file):
                    continue
                sseq_to_taxids = {}
                with gzip.open(sseq2pairs_file, 'rt') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = line.split('\t')
                        if len(parts) < 2:
                            continue
                        sseq = parts[0].strip()
                        pairs_str = parts[1]
                        # extracttaxid
                        taxids = extract_taxids_from_pairs(pairs_str)
                        if taxids:
                            sseq_to_taxids[sseq] = taxids
                            # 4. processpeptidefile
                            chunksize = 50000
                            for chunk in pd.read_csv(peptide_file, sep='\t',
                                usecols=['peptide', 'source_ids'],
                                compression='gzip',
                                chunksize=chunksize):
                            for _, row in chunk.iterrows():
                                peptide = str(row['peptide']).strip()
                                if not peptide:
                                    continue
                                source_ids = str(row['source_ids']).split(',')
                                for source_id in source_ids:
                                    source_id = source_id.strip()
                                    if not source_id:
                                        continue
                                    if source_id in id2seq_dict:
                                        sseq = id2seq_dict[source_id]
                                        if sseq in sseq_to_taxids:
                                            taxids = sseq_to_taxids[sseq]
                                            if taxids.intersection(microbes_taxids):
                                                peptides.add(peptide)
                                                break
        except Exception as e:
            # textprocesserror, text(textrowsoutputtext)
            pass
        return peptides


def extract_taxids_from_pairs(pairs_str: str) -> Set[str]:
    """frompairstextinextractalltaxid(text)"""
    taxids = set()
    if not isinstance(pairs_str, str):
        return taxids
    records = pairs_str.split(';')
    for record in records:
        record = record.strip()
        if not record:
            continue
        if 'CTAXA:' in record:
            pattern = r'CTAXA:([\d,]+)'
            matches = re.findall(pattern, record)
            for match in matches:
                for taxid in match.split(','):
                    taxid = taxid.strip()
                    if taxid and taxid.isdigit():
                        taxids.add(taxid)
                        return taxids


def main():
    """text"""
    base_dir = "/path/to/project"
    # createanalyzetext, text64text
    analyzer = PeptideSharingAnalyzer(base_dir, n_workers=64)
    # textrowsanalyze
    results_df = analyzer.run_analysis(
        cancers_to_analyze=["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG",         "KIDNEY", "BLCA", "STAD", "LIHC", "THCA", 'PanCancer'],
        max_samples=None # process all samples
)
# textresults
print("\n" + "=" * 80)
print("textresultstext:")
print("=" * 80)
print(results_df.to_string())


if __name__ == "__main__":
    main()

In [ ]:
import os
import gzip
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Set, Tuple
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
import multiprocessing as mp

class PeptideSharingAnalyzer:
    def __init__(self, base_dir: str, n_workers: int = None):
        self.base_dir = base_dir
        self.microbe_to_taxids = {}
        self.cancer_microbes = {}
        self.cancer_samples = {}
        # text, textalltext
        if n_workers is None:
            self.n_workers = mp.cpu_count()
        else:
            self.n_workers = min(n_workers, mp.cpu_count())
            print(f"text {self.n_workers} textrowstextrowsprocess")
            def extract_taxids_from_pairs(self, pairs_str: str) -> Set[str]:
                """frompairstextinextractalltaxid"""
                taxids = set()
                if not isinstance(pairs_str, str):
                    return taxids
                records = pairs_str.split(';')
                for record in records:
                    record = record.strip()
                    if not record:
                        continue
                    if 'CTAXA:' in record:
                        pattern = r'CTAXA:([\d,]+)'
                        matches = re.findall(pattern, record)
                        for match in matches:
                            for taxid in match.split(','):
                                taxid = taxid.strip()
                                if taxid and taxid.isdigit():
                                    taxids.add(taxid)
                                    return taxids
                                def load_all_data(self):
                                    """loadallrequireddata"""
                                    taxid_file = os.path.join(self.base_dir, "data/genome_annotation/V7_taxid_hierarchy.txt")
                                    self._load_taxid_mapping(taxid_file)
                                    microbes_dir = os.path.join(self.base_dir, "results_V3/summary/sharing_rate")
                                    self._load_shared_microbes(microbes_dir)
                                    sample_info_file = os.path.join(self.base_dir, "data_V3/meta/cancer_status_sample_1897.csv")
                                    self._load_sample_info(sample_info_file)
                                    def _load_taxid_mapping(self, taxid_file: str):
                                        """loadtaxidmapping"""
                                        print(f"loadtaxidmappingfile: {taxid_file}")
                                        df = pd.read_csv(taxid_file, sep='\t')
                                        for _, row in df.iterrows():
                                            species = str(row['species_scientific_name'])
                                            processed_name = species.replace(' ', '_').replace('/', '_')
                                            taxid = str(row['tax_id'])
                                            if processed_name not in self.microbe_to_taxids:
                                                self.microbe_to_taxids[processed_name] = []
                                                if taxid not in self.microbe_to_taxids[processed_name]:
                                                    self.microbe_to_taxids[processed_name].append(taxid)
                                                    print(f"loadtext {len(self.microbe_to_taxids)} microbestaxidmapping")
                                                    def _load_shared_microbes(self, microbes_dir: str):
                                                        """loadtextmicrobe list(fromsharing_ratedirectory)"""
                                                        print(f"loadtextmicrobe list: {microbes_dir}")
                                                        self.cancer_microbes = {}
                                                        for filename in os.listdir(microbes_dir):
                                                            if filename.endswith('_shared_microbes.txt'):
                                                                cancer = filename.split('_shared_microbes.txt')[0]
                                                                filepath = os.path.join(microbes_dir, filename)
                                                                with open(filepath, 'r') as f:
                                                                    microbes = [line.strip() for line in f if line.strip()]
                                                                    self.cancer_microbes[cancer] = microbes
                                                                    print(f" {cancer}: {len(microbes)} sharedmicrobe")
                                                                    def _load_sample_info(self, sample_info_file: str):
                                                                        """loadsampleinformation"""
                                                                        print(f"loadsampleinformation: {sample_info_file}")
                                                                        df = pd.read_csv(sample_info_file)
                                                                        for _, row in df.iterrows():
                                                                            cancer = row['Cancer_Type']
                                                                            status = row['Status']
                                                                            sample = row['Sample']
                                                                            if cancer not in self.cancer_samples:
                                                                                self.cancer_samples[cancer] = {'Normal': [], 'Tumor': []}
                                                                                self.cancer_samples[cancer][status].append(sample)
                                                                                total_samples = 0
                                                                                for cancer, status_dict in self.cancer_samples.items():
                                                                                    normal_count = len(status_dict.get('Normal', []))
                                                                                    tumor_count = len(status_dict.get('Tumor', []))
                                                                                    total_samples += normal_count + tumor_count
                                                                                    print(f" {cancer}: Normal={normal_count}, Tumor={tumor_count}")
                                                                                    print(f"text {total_samples} samples")
                                                                                    def get_microbes_taxids_for_cancer(self, cancer: str) -> Set[str]:
                                                                                        """textcancer typesharedmicrobetaxidtext"""
                                                                                        if cancer not in self.cancer_microbes:
                                                                                            return set()
                                                                                        microbes_taxids = set()
                                                                                        not_found_count = 0
                                                                                        for microbe in self.cancer_microbes[cancer]:
                                                                                            if microbe in self.microbe_to_taxids:
                                                                                                microbes_taxids.update(self.microbe_to_taxids[microbe])
                                                                                            else:
                                                                                                original_name = microbe.replace('_', ' ')
                                                                                                if original_name in self.microbe_to_taxids:
                                                                                                    microbes_taxids.update(self.microbe_to_taxids[original_name])
                                                                                                else:
                                                                                                    not_found_count += 1
                                                                                                    if not_found_count > 0:
                                                                                                        print(f" {cancer}: {not_found_count} microbesnot foundtaxidmapping")
                                                                                                        return microbes_taxids
                                                                                                    def save_peptide_lists(self, cancer: str, normal_peptides: Set[str],                                                                                                         tumor_peptides: Set[str], output_base_dir: str):
                                                                                                    """savetextcancer typeallpeptidelist"""
                                                                                                    cancer_dir = os.path.join(output_base_dir, cancer)
                                                                                                    os.makedirs(cancer_dir, exist_ok=True)
                                                                                                    # calculatetextpeptides
                                                                                                    shared_peptides = normal_peptides & tumor_peptides
                                                                                                    normal_unique = normal_peptides - tumor_peptides
                                                                                                    tumor_unique = tumor_peptides - normal_peptides
                                                                                                    # textfortextafterlist
                                                                                                    peptide_lists = {
                                                                                                        'Normal_peptides.txt': sorted(normal_peptides),
                                                                                                        'Tumor_peptides.txt': sorted(tumor_peptides),
                                                                                                        'Shared_peptides.txt': sorted(shared_peptides),
                                                                                                        'Normal_unique_peptides.txt': sorted(normal_unique),
                                                                                                        'Tumor_unique_peptides.txt': sorted(tumor_unique)
                                                                                                    }
                                                                                                    # savefile
                                                                                                    for filename, peptide_list in peptide_lists.items():
                                                                                                        filepath = os.path.join(cancer_dir, filename)
                                                                                                        with open(filepath, 'w') as f:
                                                                                                            f.write('\n'.join(peptide_list))
                                                                                                            return cancer_dir, shared_peptides, normal_unique, tumor_unique
                                                                                                        def analyze_cancer(self, cancer: str, output_base_dir: str, max_samples_per_status: int = None) -> Dict:
                                                                                                            """analyzetextcancer typepeptidetextsavepeptidelist"""
                                                                                                            print(f"\nanalyzecancer type: {cancer}")
                                                                                                            # 1. textsharedmicrobetaxid
                                                                                                            microbes_taxids = self.get_microbes_taxids_for_cancer(cancer)
                                                                                                            if not microbes_taxids:
                                                                                                                print(f" error: not foundwithtexttaxid")
                                                                                                                return None
                                                                                                            print(f" sharedmicrobetexttaxidcount: {len(microbes_taxids)}")
                                                                                                            # 2. textsample columntext
                                                                                                            if cancer == 'PanCancer':
                                                                                                                normal_samples = []
                                                                                                                tumor_samples = []
                                                                                                                for c_type, status_dict in self.cancer_samples.items():
                                                                                                                    if c_type != 'PanCancer':
                                                                                                                        normal_samples.extend(status_dict.get('Normal', []))
                                                                                                                        tumor_samples.extend(status_dict.get('Tumor', []))
                                                                                                                    else:
                                                                                                                        if cancer not in self.cancer_samples:
                                                                                                                            print(f" error: cancer type {cancer} textsampleinformationinnot found")
                                                                                                                            return None
                                                                                                                        if max_samples_per_status:
                                                                                                                            normal_samples = self.cancer_samples[cancer].get('Normal', [])[:max_samples_per_status]
                                                                                                                            tumor_samples = self.cancer_samples[cancer].get('Tumor', [])[:max_samples_per_status]
                                                                                                                        else:
                                                                                                                            normal_samples = self.cancer_samples[cancer].get('Normal', [])
                                                                                                                            tumor_samples = self.cancer_samples[cancer].get('Tumor', [])
                                                                                                                            print(f" Normalsample: {len(normal_samples)} ")
                                                                                                                            print(f" Tumorsample: {len(tumor_samples)} ")
                                                                                                                            # 3. textrowsProcess sample
                                                                                                                            print(" textrowsprocessNormalsamplepeptide...")
                                                                                                                            normal_peptides = self._process_samples_parallel(normal_samples, microbes_taxids, "Normal")
                                                                                                                            print(" textrowsprocessTumorsamplepeptide...")
                                                                                                                            tumor_peptides = self._process_samples_parallel(tumor_samples, microbes_taxids, "Tumor")
                                                                                                                            # 4. savepeptidelist
                                                                                                                            saved_dir, shared_peptides, normal_unique, tumor_unique = self.save_peptide_lists(
                                                                                                                                cancer, normal_peptides, tumor_peptides, output_base_dir
)
# 5. calculatestatistics
total = len(shared_peptides) + len(normal_unique) + len(tumor_unique)
if total > 0:
    shared_percent = (len(shared_peptides) / total * 100)
    normal_percent = (len(normal_unique) / total * 100)
    tumor_percent = (len(tumor_unique) / total * 100)
else:
    shared_percent = normal_percent = tumor_percent = 0.0
    def format_percent(value):
        return f"{value:.1f}%"
    result = {
        'Cancer': cancer,
        'Normal_Samples': len(normal_samples),
        'Tumor_Samples': len(tumor_samples),
        'Shared_Microbes': len(self.cancer_microbes.get(cancer, [])),
        'Total_normal_peptides': len(normal_peptides),
        'Total_tumor_peptides': len(tumor_peptides),
        'Shared_peptides': len(shared_peptides),
        'Shared_percentage': format_percent(shared_percent),
        'Normal_unique_peptides': len(normal_unique),
        'Normal_unique_percentage': format_percent(normal_percent),
        'Tumor_unique_peptides': len(tumor_unique),
        'Tumor_unique_percentage': format_percent(tumor_percent)
    }
    print(f" results:")
    print(f" Normaltotal peptides: {len(normal_peptides):,}")
    print(f" Tumortotal peptides: {len(tumor_peptides):,}")
    print(f" Normal unique: {len(normal_unique):,} ({result['Normal_unique_percentage']})")
    print(f" Tumor unique: {len(tumor_unique):,} ({result['Tumor_unique_percentage']})")
    print(f" shared peptides: {len(shared_peptides):,} ({result['Shared_percentage']})")
    print(f" OK Peptidelistsavedto: {saved_dir}")
    return result
    def _process_samples_parallel(self, samples: List[str], microbes_taxids: Set[str],         sample_type: str) -> Set[str]:
    """textrowsprocesstextsamples"""
    all_peptides = set()
    completed = 0
    total = len(samples)
    # createtext
    with ProcessPoolExecutor(max_workers=self.n_workers) as executor:
        # textalltext
        future_to_sample = {
            executor.submit(process_single_sample, sample, self.base_dir, microbes_taxids): sample
            for sample in samples
        }
        # textresults
        for future in as_completed(future_to_sample):
            sample = future_to_sample[future]
            completed += 1
            try:
                peptides = future.result()
                all_peptides.update(peptides)
                if completed % 10 == 0 or completed == total:
                    print(f" [{completed}/{total}] completed {sample_type} sampleprocess, "
                        f"cumulativepeptides={len(all_peptides):,}")
            except Exception as e:
                print(f" [{completed}/{total}] process {sample_type} sample {sample} text: {str(e)}")
                return all_peptides
            def run_analysis(self, cancers_to_analyze=None, max_samples=None):
                """textrowstextanalyze"""
                print("=" * 80)
                print(" startpeptidetextanalyze(textsharedmicrobe, textrowstext)")
                print("=" * 80)
                print("\nanalyzetext:")
                print(" 1. fromsharing_ratedirectoryreadtextcancer typesharedmicrobe")
                print(" 2. extracttextmicrobetextNormal/Tumorsampleinpeptides")
                print(" 3. statisticstextsavepeptidetext")
                print("=" * 80)
                self.load_all_data()
                if cancers_to_analyze is None:
                    cancers_to_analyze = list(self.cancer_microbes.keys())
                    # output directory
                    output_base_dir = os.path.join(self.base_dir,                         "results_V3/summary/shareing_rate_peptide/public/with_speciesINFO")
                    os.makedirs(output_base_dir, exist_ok=True)
                    results = []
                    for cancer in cancers_to_analyze:
                        result = self.analyze_cancer(cancer, output_base_dir, max_samples_per_status=max_samples)
                        if result:
                            results.append(result)
                            df = pd.DataFrame(results)
                            # savetextstatistics
                            output_file = os.path.join(output_base_dir, "peptide_sharing_results.csv")
                            df.to_csv(output_file, index=False)
                            print(f"\nOK textstatisticsresultssavedto: {output_file}")
                            # savedetailedtext
                            summary_file = os.path.join(output_base_dir, "peptide_sharing_summary.txt")
                            with open(summary_file, 'w') as f:
                                f.write("Peptide-sharing analysis summary(textsharedmicrobe)\n")
                                f.write("=" * 80 + "\n\n")
                                for _, row in df.iterrows():
                                    f.write(f"cancer type: {row['Cancer']}\n")
                                    f.write(f" sample count: Normal={row['Normal_Samples']}, Tumor={row['Tumor_Samples']}\n")
                                    f.write(f" shared microbe count: {row['Shared_Microbes']}\n")
                                    f.write(f" Normaltotal peptides: {row['Total_normal_peptides']}\n")
                                    f.write(f" Tumortotal peptides: {row['Total_tumor_peptides']}\n")
                                    f.write(f" shared peptides: {row['Shared_peptides']} ({row['Shared_percentage']})\n")
                                    f.write(f" Normal unique: {row['Normal_unique_peptides']} ({row['Normal_unique_percentage']})\n")
                                    f.write(f" Tumor unique: {row['Tumor_unique_peptides']} ({row['Tumor_unique_percentage']})\n")
                                    f.write("-" * 60 + "\n")
                                    print(f"OK detailedstatisticssavedto: {summary_file}")
                                    print("\n" + "=" * 80)
                                    print("filetext:")
                                    print(" eachcancer type/textwithtextfiletext, text5files:")
                                    print(" - Normal_peptides.txt: Normal peptidetext")
                                    print(" - Tumor_peptides.txt: Tumor peptidetext")
                                    print(" - Shared_peptides.txt: textpeptides")
                                    print(" - Normal_unique_peptides.txt: Normal uniquepeptides")
                                    print(" - Tumor_unique_peptides.txt: Tumor uniquepeptides")
                                    print("=" * 80)
                                    print("analyzecompleted!")
                                    print("=" * 80)
                                    return df


# text, textrowsprocess
def process_single_sample(sample: str, base_dir: str, microbes_taxids: Set[str]) -> Set[str]:
    """
    processtextsamplestext(text)
    text:
    sample: sampleID
    base_dir: textdirectory
    microbes_taxids: textmicrobetaxidtext
    return:
    Set[str]: peptide sequencetext
    """
    peptides = set()
    # processHLAIandHLAII
    for hla_type in ['HLAI', 'HLAII']:
        try:
            # 1. textpeptidefilepath
            bindlevel_dir = os.path.join(base_dir, "data_V3/peptides/pathseq_reads_peptides_public/filtered_out/BindLevel")
            if hla_type == 'HLAI':
                peptide_file = os.path.join(bindlevel_dir, "HLAI", f"{sample}.pHLA_I.tsv.gz")
            else:
                peptide_file = os.path.join(bindlevel_dir, "HLAII", f"{sample}.pHLA_II.tsv.gz")
                if not os.path.exists(peptide_file):
                    continue
                # 2. readsource_idtosseqmapping
                phla_dir = os.path.join(base_dir, "data_V3/peptides/pathseq_reads_peptides_public/pHLA_bare")
                id2seq_file = os.path.join(phla_dir, sample, f"{sample}.id2seq.tsv.gz")
                if not os.path.exists(id2seq_file):
                    continue
                df_id2seq = pd.read_csv(id2seq_file, sep='\t', usecols=['Identity', 'sseq'], compression='gzip')
                id2seq_dict = dict(zip(df_id2seq['Identity'], df_id2seq['sseq']))
                # 3. readsseqtotaxidmapping
                sseq2pairs_file = os.path.join(phla_dir, sample, f"{sample}.sseq2pairs.tsv.gz")
                if not os.path.exists(sseq2pairs_file):
                    continue
                sseq_to_taxids = {}
                with gzip.open(sseq2pairs_file, 'rt') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = line.split('\t')
                        if len(parts) < 2:
                            continue
                        sseq = parts[0].strip()
                        pairs_str = parts[1]
                        # extracttaxid
                        taxids = extract_taxids_from_pairs(pairs_str)
                        if taxids:
                            sseq_to_taxids[sseq] = taxids
                            # 4. processpeptidefile
                            chunksize = 50000
                            for chunk in pd.read_csv(peptide_file, sep='\t',
                                usecols=['peptide', 'source_ids'],
                                compression='gzip',
                                chunksize=chunksize):
                            for _, row in chunk.iterrows():
                                peptide = str(row['peptide']).strip()
                                if not peptide:
                                    continue
                                source_ids = str(row['source_ids']).split(',')
                                for source_id in source_ids:
                                    source_id = source_id.strip()
                                    if not source_id:
                                        continue
                                    if source_id in id2seq_dict:
                                        sseq = id2seq_dict[source_id]
                                        if sseq in sseq_to_taxids:
                                            taxids = sseq_to_taxids[sseq]
                                            if taxids.intersection(microbes_taxids):
                                                peptides.add(peptide)
                                                break
        except Exception as e:
            # textprocesserror, text(textrowsoutputtext)
            pass
        return peptides


def extract_taxids_from_pairs(pairs_str: str) -> Set[str]:
    """frompairstextinextractalltaxid(text)"""
    taxids = set()
    if not isinstance(pairs_str, str):
        return taxids
    records = pairs_str.split(';')
    for record in records:
        record = record.strip()
        if not record:
            continue
        if 'CTAXA:' in record:
            pattern = r'CTAXA:([\d,]+)'
            matches = re.findall(pattern, record)
            for match in matches:
                for taxid in match.split(','):
                    taxid = taxid.strip()
                    if taxid and taxid.isdigit():
                        taxids.add(taxid)
                        return taxids


def main():
    """text"""
    base_dir = "/path/to/project"
    # createanalyzetext, text64text
    analyzer = PeptideSharingAnalyzer(base_dir, n_workers=64)
    # textrowsanalyze
    results_df = analyzer.run_analysis(
        cancers_to_analyze=["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG",         "KIDNEY", "BLCA", "STAD", "LIHC", "THCA", 'PanCancer'],
        max_samples=None # process all samples
)
# textresults
print("\n" + "=" * 80)
print("textresultstext:")
print("=" * 80)
print(results_df.to_string())


if __name__ == "__main__":
    main()

In [ ]:
import os
import gzip
import pandas as pd
from collections import defaultdict
from typing import Dict, List, Set, Tuple
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial
import multiprocessing as mp

class PeptideSharingAnalyzer:
    def __init__(self, base_dir: str, n_workers: int = None):
        self.base_dir = base_dir
        self.microbe_to_taxids = {}
        self.cancer_microbes = {}
        self.cancer_samples = {}
        # text, textalltext
        if n_workers is None:
            self.n_workers = mp.cpu_count()
        else:
            self.n_workers = min(n_workers, mp.cpu_count())
            print(f"text: text {self.n_workers} text(availableCPU: {mp.cpu_count()}text)")
            def extract_taxids_from_pairs(self, pairs_str: str) -> Set[str]:
                """frompairstextinextractalltaxid"""
                taxids = set()
                if not isinstance(pairs_str, str):
                    return taxids
                records = pairs_str.split(';')
                for record in records:
                    record = record.strip()
                    if not record:
                        continue
                    if 'CTAXA:' in record:
                        pattern = r'CTAXA:([\d,]+)'
                        matches = re.findall(pattern, record)
                        for match in matches:
                            for taxid in match.split(','):
                                taxid = taxid.strip()
                                if taxid and taxid.isdigit():
                                    taxids.add(taxid)
                                    return taxids
                                def load_all_data(self):
                                    """loadallrequireddata"""
                                    taxid_file = os.path.join(self.base_dir, "data/genome_annotation/V7_taxid_hierarchy.txt")
                                    self._load_taxid_mapping(taxid_file)
                                    microbes_dir = os.path.join(self.base_dir, "results_V3/summary/sharing_rate")
                                    self._load_shared_microbes(microbes_dir)
                                    sample_info_file = os.path.join(self.base_dir, "data_V3/meta/cancer_status_sample_1897.csv")
                                    self._load_sample_info(sample_info_file)
                                    def _load_taxid_mapping(self, taxid_file: str):
                                        """loadtaxidmapping"""
                                        print(f"\n[dataload] taxidmappingfile: {taxid_file}")
                                        df = pd.read_csv(taxid_file, sep='\t')
                                        for _, row in df.iterrows():
                                            species = str(row['species_scientific_name'])
                                            processed_name = species.replace(' ', '_').replace('/', '_')
                                            taxid = str(row['tax_id'])
                                            if processed_name not in self.microbe_to_taxids:
                                                self.microbe_to_taxids[processed_name] = []
                                                if taxid not in self.microbe_to_taxids[processed_name]:
                                                    self.microbe_to_taxids[processed_name].append(taxid)
                                                    print(f" OK loadtext {len(self.microbe_to_taxids)} microbestaxidmapping")
                                                    def _load_shared_microbes(self, microbes_dir: str):
                                                        """loadtextmicrobe list(fromsharing_ratedirectory)"""
                                                        print(f"\n[dataload] textmicrobe list: {microbes_dir}")
                                                        self.cancer_microbes = {}
                                                        for filename in os.listdir(microbes_dir):
                                                            if filename.endswith('_shared_microbes.txt'):
                                                                cancer = filename.split('_shared_microbes.txt')[0]
                                                                filepath = os.path.join(microbes_dir, filename)
                                                                with open(filepath, 'r') as f:
                                                                    microbes = [line.strip() for line in f if line.strip()]
                                                                    self.cancer_microbes[cancer] = microbes
                                                                    print(f" {cancer}: {len(microbes)} sharedmicrobe")
                                                                    def _load_sample_info(self, sample_info_file: str):
                                                                        """loadsampleinformation"""
                                                                        print(f"\n[dataload] sampleinformation: {sample_info_file}")
                                                                        df = pd.read_csv(sample_info_file)
                                                                        for _, row in df.iterrows():
                                                                            cancer = row['Cancer_Type']
                                                                            status = row['Status']
                                                                            sample = row['Sample']
                                                                            if cancer not in self.cancer_samples:
                                                                                self.cancer_samples[cancer] = {'Normal': [], 'Tumor': []}
                                                                                self.cancer_samples[cancer][status].append(sample)
                                                                                total_samples = 0
                                                                                for cancer, status_dict in self.cancer_samples.items():
                                                                                    normal_count = len(status_dict.get('Normal', []))
                                                                                    tumor_count = len(status_dict.get('Tumor', []))
                                                                                    total_samples += normal_count + tumor_count
                                                                                    print(f" {cancer}: Normal={normal_count}, Tumor={tumor_count}")
                                                                                    print(f" text: {total_samples} samples")
                                                                                    def get_microbes_taxids_for_cancer(self, cancer: str) -> Set[str]:
                                                                                        """textcancer typesharedmicrobetaxidtext"""
                                                                                        if cancer not in self.cancer_microbes:
                                                                                            return set()
                                                                                        microbes_taxids = set()
                                                                                        not_found_count = 0
                                                                                        for microbe in self.cancer_microbes[cancer]:
                                                                                            if microbe in self.microbe_to_taxids:
                                                                                                microbes_taxids.update(self.microbe_to_taxids[microbe])
                                                                                            else:
                                                                                                original_name = microbe.replace('_', ' ')
                                                                                                if original_name in self.microbe_to_taxids:
                                                                                                    microbes_taxids.update(self.microbe_to_taxids[original_name])
                                                                                                else:
                                                                                                    not_found_count += 1
                                                                                                    if not_found_count > 0:
                                                                                                        print(f" warning: {not_found_count} microbesnot foundtaxidmapping")
                                                                                                        return microbes_taxids
                                                                                                    def save_peptide_lists(self, cancer: str, normal_peptides: Set[str],                                                                                                         tumor_peptides: Set[str], output_base_dir: str):
                                                                                                    """savetextcancer typeallpeptidelist"""
                                                                                                    cancer_dir = os.path.join(output_base_dir, cancer)
                                                                                                    os.makedirs(cancer_dir, exist_ok=True)
                                                                                                    # calculatetextpeptides
                                                                                                    shared_peptides = normal_peptides & tumor_peptides
                                                                                                    normal_unique = normal_peptides - tumor_peptides
                                                                                                    tumor_unique = tumor_peptides - normal_peptides
                                                                                                    # textfortextafterlist
                                                                                                    peptide_lists = {
                                                                                                        'Normal_peptides.txt': sorted(normal_peptides),
                                                                                                        'Tumor_peptides.txt': sorted(tumor_peptides),
                                                                                                        'Shared_peptides.txt': sorted(shared_peptides),
                                                                                                        'Normal_unique_peptides.txt': sorted(normal_unique),
                                                                                                        'Tumor_unique_peptides.txt': sorted(tumor_unique)
                                                                                                    }
                                                                                                    # savefile
                                                                                                    for filename, peptide_list in peptide_lists.items():
                                                                                                        filepath = os.path.join(cancer_dir, filename)
                                                                                                        with open(filepath, 'w') as f:
                                                                                                            f.write('\n'.join(peptide_list))
                                                                                                            return cancer_dir, shared_peptides, normal_unique, tumor_unique
                                                                                                        def analyze_cancer(self, cancer: str, output_base_dir: str, max_samples_per_status: int = None) -> Dict:
                                                                                                            """analyzetextcancer typepeptidetextsavepeptidelist"""
                                                                                                            print(f"\n{'='*80}")
                                                                                                            print(f"analyzecancer type: {cancer}")
                                                                                                            print(f"{'='*80}")
                                                                                                            # 1. textsharedmicrobetaxid
                                                                                                            microbes_taxids = self.get_microbes_taxids_for_cancer(cancer)
                                                                                                            if not microbes_taxids:
                                                                                                                print(f" Failed error: not foundwithtexttaxid")
                                                                                                                return None
                                                                                                            print(f" OK sharedmicrobetexttaxidcount: {len(microbes_taxids)}")
                                                                                                            # 2. textsample columntext
                                                                                                            if cancer == 'PanCancer':
                                                                                                                normal_samples = []
                                                                                                                tumor_samples = []
                                                                                                                for c_type, status_dict in self.cancer_samples.items():
                                                                                                                    if c_type != 'PanCancer':
                                                                                                                        normal_samples.extend(status_dict.get('Normal', []))
                                                                                                                        tumor_samples.extend(status_dict.get('Tumor', []))
                                                                                                                    else:
                                                                                                                        if cancer not in self.cancer_samples:
                                                                                                                            print(f" Failed error: cancer type {cancer} textsampleinformationinnot found")
                                                                                                                            return None
                                                                                                                        if max_samples_per_status:
                                                                                                                            normal_samples = self.cancer_samples[cancer].get('Normal', [])[:max_samples_per_status]
                                                                                                                            tumor_samples = self.cancer_samples[cancer].get('Tumor', [])[:max_samples_per_status]
                                                                                                                        else:
                                                                                                                            normal_samples = self.cancer_samples[cancer].get('Normal', [])
                                                                                                                            tumor_samples = self.cancer_samples[cancer].get('Tumor', [])
                                                                                                                            print(f" OK Normalsample: {len(normal_samples)} ")
                                                                                                                            print(f" OK Tumorsample: {len(tumor_samples)} ")
                                                                                                                            # 3. textrowsProcess sample
                                                                                                                            print(f"\n [textrowsprocess] Normalsamplepeptideextract...")
                                                                                                                            normal_peptides = self._process_samples_parallel(normal_samples, microbes_taxids, "Normal")
                                                                                                                            print(f"\n [textrowsprocess] Tumorsamplepeptideextract...")
                                                                                                                            tumor_peptides = self._process_samples_parallel(tumor_samples, microbes_taxids, "Tumor")
                                                                                                                            # 4. savepeptidelist
                                                                                                                            print(f"\n [save data] textpeptidelist...")
                                                                                                                            saved_dir, shared_peptides, normal_unique, tumor_unique = self.save_peptide_lists(
                                                                                                                                cancer, normal_peptides, tumor_peptides, output_base_dir
)
# 5. calculatestatistics
total = len(shared_peptides) + len(normal_unique) + len(tumor_unique)
if total > 0:
    shared_percent = (len(shared_peptides) / total * 100)
    normal_percent = (len(normal_unique) / total * 100)
    tumor_percent = (len(tumor_unique) / total * 100)
else:
    shared_percent = normal_percent = tumor_percent = 0.0
    def format_percent(value):
        return f"{value:.1f}%"
    result = {
        'Cancer': cancer,
        'Normal_Samples': len(normal_samples),
        'Tumor_Samples': len(tumor_samples),
        'Shared_Microbes': len(self.cancer_microbes.get(cancer, [])),
        'Total_normal_peptides': len(normal_peptides),
        'Total_tumor_peptides': len(tumor_peptides),
        'Shared_peptides': len(shared_peptides),
        'Shared_percentage': format_percent(shared_percent),
        'Normal_unique_peptides': len(normal_unique),
        'Normal_unique_percentage': format_percent(normal_percent),
        'Tumor_unique_peptides': len(tumor_unique),
        'Tumor_unique_percentage': format_percent(tumor_percent)
    }
    print(f"\n [statisticsresults]")
    print(f" Normaltotal peptides: {len(normal_peptides):>10,}")
    print(f" Tumortotal peptides: {len(tumor_peptides):>10,}")
    print(f" Normal unique: {len(normal_unique):>10,} ({result['Normal_unique_percentage']})")
    print(f" Tumor unique: {len(tumor_unique):>10,} ({result['Tumor_unique_percentage']})")
    print(f" shared peptides: {len(shared_peptides):>10,} ({result['Shared_percentage']})")
    print(f" OK savedto: {saved_dir}")
    return result
    def _process_samples_parallel(self, samples: List[str], microbes_taxids: Set[str],         sample_type: str) -> Set[str]:
    """textrowsprocesstextsamples(text)"""
    all_peptides = set()
    completed = 0
    total = len(samples)
    if total == 0:
        return all_peptides
    print(f" text {self.n_workers} textprocess {total} samples...")
    # createtext
    with ProcessPoolExecutor(max_workers=self.n_workers) as executor:
        # textalltext
        future_to_sample = {
            executor.submit(process_single_sample, sample, self.base_dir, microbes_taxids): sample
            for sample in samples
        }
        # textresults
        for future in as_completed(future_to_sample):
            sample = future_to_sample[future]
            completed += 1
            try:
                peptides = future.result()
                all_peptides.update(peptides)
                # text: textcompleted5%or10samplesUpdatetext
                update_interval = max(10, total // 20)
                if completed % update_interval == 0 or completed == total:
                    progress = (completed / total) * 100
                    print(f" text: [{completed:>4}/{total}] {progress:>5.1f}% | "
                        f"cumulativepeptides: {len(all_peptides):>8,}")
            except Exception as e:
                print(f" Failed sample {sample} processfailed: {str(e)}")
                print(f" OK completed {sample_type} sampleprocess, text {len(all_peptides):,} peptides")
                return all_peptides
            def run_analysis(self, cancers_to_analyze=None, max_samples=None):
                """textrowstextanalyze"""
                print("\n" + "=" * 80)
                print(" peptidetextanalyze(textsharedmicrobe, textrowstext)")
                print("=" * 80)
                print("\nanalyzetext:")
                print(" 1. fromsharing_ratedirectoryreadtextcancer typesharedmicrobe")
                print(" 2. extracttextmicrobetextNormal/Tumorsampleinpeptides")
                print(" 3. textrowsprocesstextanalyze")
                print(" 4. statisticstextsavepeptidetext")
                print("=" * 80)
                self.load_all_data()
                if cancers_to_analyze is None:
                    cancers_to_analyze = list(self.cancer_microbes.keys())
                    # output directory
                    output_base_dir = os.path.join(self.base_dir,                         "results_V3/summary/shareing_rate_peptide/public/with_speciesINFO")
                    os.makedirs(output_base_dir, exist_ok=True)
                    results = []
                    print("\n" + "=" * 80)
                    print("startanalyzetextcancer type...")
                    print("=" * 80)
                    for i, cancer in enumerate(cancers_to_analyze, 1):
                        print(f"\ntext: [{i}/{len(cancers_to_analyze)}]")
                        result = self.analyze_cancer(cancer, output_base_dir, max_samples_per_status=max_samples)
                        if result:
                            results.append(result)
                            df = pd.DataFrame(results)
                            # savetextstatistics
                            print("\n" + "=" * 80)
                            print("saveanalyzeresults...")
                            print("=" * 80)
                            output_file = os.path.join(output_base_dir, "peptide_sharing_results.csv")
                            df.to_csv(output_file, index=False)
                            print(f" OK textstatistics: {output_file}")
                            # savedetailedtext
                            summary_file = os.path.join(output_base_dir, "peptide_sharing_summary.txt")
                            with open(summary_file, 'w') as f:
                                f.write("Peptide-sharing analysis summary(textsharedmicrobe)\n")
                                f.write("=" * 80 + "\n\n")
                                for _, row in df.iterrows():
                                    f.write(f"cancer type: {row['Cancer']}\n")
                                    f.write(f" sample count: Normal={row['Normal_Samples']}, Tumor={row['Tumor_Samples']}\n")
                                    f.write(f" shared microbe count: {row['Shared_Microbes']}\n")
                                    f.write(f" Normaltotal peptides: {row['Total_normal_peptides']}\n")
                                    f.write(f" Tumortotal peptides: {row['Total_tumor_peptides']}\n")
                                    f.write(f" shared peptides: {row['Shared_peptides']} ({row['Shared_percentage']})\n")
                                    f.write(f" Normal unique: {row['Normal_unique_peptides']} ({row['Normal_unique_percentage']})\n")
                                    f.write(f" Tumor unique: {row['Tumor_unique_peptides']} ({row['Tumor_unique_percentage']})\n")
                                    f.write("-" * 60 + "\n")
                                    print(f" OK detailedtext: {summary_file}")
                                    print("\n" + "=" * 80)
                                    print("filetext:")
                                    print(" eachcancer type/textwithtextfiletext, text5files:")
                                    print(" - Normal_peptides.txt: Normal peptidetext")
                                    print(" - Tumor_peptides.txt: Tumor peptidetext")
                                    print(" - Shared_peptides.txt: textpeptides")
                                    print(" - Normal_unique_peptides.txt: Normal uniquepeptides")
                                    print(" - Tumor_unique_peptides.txt: Tumor uniquepeptides")
                                    print("=" * 80)
                                    print("OK analyzecompleted!")
                                    print("=" * 80)
                                    return df


# text, textrowsprocess
def process_single_sample(sample: str, base_dir: str, microbes_taxids: Set[str]) -> Set[str]:
    """
    processtextsamplestext(text)
    text: text, textprocesstext
    """
    peptides = set()
    # processHLAIandHLAII
    for hla_type in ['HLAI', 'HLAII']:
        try:
            # 1. textpeptidefilepath
            bindlevel_dir = os.path.join(base_dir, "data_V3/peptides/pathseq_reads_peptides_public/filtered_out/BindLevel")
            if hla_type == 'HLAI':
                peptide_file = os.path.join(bindlevel_dir, "HLAI", f"{sample}.pHLA_I.tsv.gz")
            else:
                peptide_file = os.path.join(bindlevel_dir, "HLAII", f"{sample}.pHLA_II.tsv.gz")
                if not os.path.exists(peptide_file):
                    continue
                # 2. readsource_idtosseqmapping
                phla_dir = os.path.join(base_dir, "data_V3/peptides/pathseq_reads_peptides_public/pHLA_bare")
                id2seq_file = os.path.join(phla_dir, sample, f"{sample}.id2seq.tsv.gz")
                if not os.path.exists(id2seq_file):
                    continue
                df_id2seq = pd.read_csv(id2seq_file, sep='\t', usecols=['Identity', 'sseq'], compression='gzip')
                id2seq_dict = dict(zip(df_id2seq['Identity'], df_id2seq['sseq']))
                del df_id2seq # release memory
                # 3. readsseqtotaxidmapping
                sseq2pairs_file = os.path.join(phla_dir, sample, f"{sample}.sseq2pairs.tsv.gz")
                if not os.path.exists(sseq2pairs_file):
                    continue
                sseq_to_taxids = {}
                with gzip.open(sseq2pairs_file, 'rt') as f:
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        parts = line.split('\t')
                        if len(parts) < 2:
                            continue
                        sseq = parts[0].strip()
                        pairs_str = parts[1]
                        # extracttaxid
                        taxids = extract_taxids_from_pairs(pairs_str)
                        if taxids:
                            sseq_to_taxids[sseq] = taxids
                            # 4. processpeptidefile(textreadtext)
                            chunksize = 100000 # increase chunk size to improve efficiency
                            for chunk in pd.read_csv(peptide_file, sep='\t',
                                usecols=['peptide', 'source_ids'],
                                compression='gzip',
                                chunksize=chunksize):
                            for _, row in chunk.iterrows():
                                peptide = str(row['peptide']).strip()
                                if not peptide:
                                    continue
                                source_ids = str(row['source_ids']).split(',')
                                for source_id in source_ids:
                                    source_id = source_id.strip()
                                    if not source_id:
                                        continue
                                    sseq = id2seq_dict.get(source_id)
                                    if sseq:
                                        taxids = sseq_to_taxids.get(sseq)
                                        if taxids and taxids.intersection(microbes_taxids):
                                            peptides.add(peptide)
                                            break
        except Exception as e:
            # textprocesserror(textrowsoutputtext)
            pass
        return peptides


def extract_taxids_from_pairs(pairs_str: str) -> Set[str]:
    """frompairstextinextractalltaxid(text)"""
    taxids = set()
    if not isinstance(pairs_str, str):
        return taxids
    records = pairs_str.split(';')
    for record in records:
        record = record.strip()
        if not record:
            continue
        if 'CTAXA:' in record:
            pattern = r'CTAXA:([\d,]+)'
            matches = re.findall(pattern, record)
            for match in matches:
                for taxid in match.split(','):
                    taxid = taxid.strip()
                    if taxid and taxid.isdigit():
                        taxids.add(taxid)
                        return taxids


def main():
    """text"""
    base_dir = "/path/to/project"
    # createanalyzetext, text64text
    analyzer = PeptideSharingAnalyzer(base_dir, n_workers=64)
    # textrowsanalyze
    results_df = analyzer.run_analysis(
        cancers_to_analyze=["OSCC", "BRCA", "PAAD", "CESC", "CRC", "LUNG",         "KIDNEY", "BLCA", "STAD", "LIHC", "THCA", 'PanCancer'],
        max_samples=None # process all samples
)
# textresults
print("\n" + "=" * 80)
print("textresultstext:")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)


if __name__ == "__main__":
    main()